# Train the "Hey UNO" and "Hello UNO" wake words

This notebook builds both wake-word models from openly licensed data, all of it allowed for commercial use, and packages them for the app.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. You need about **10 GB free in Google Drive**: everything is saved to `MyDrive/wakeword/<run>` (default `uno-v1`) so nothing is lost if Colab disconnects.

**To run:** *Runtime → Run all*. Expect 5–7 hours in total; the free tier may stop sooner, which is fine because every step resumes. If the session drops, reconnect and *Run all* again; finished steps are skipped and long steps resume where they stopped.

| Step | What it does | Time on a T4 (approx.) |
| --- | --- | --- |
| Setup | Downloads the feature models, the Piper voice and room recordings | 2 min |
| Preview | Lets you listen to how the voices say each phrase | 1 min |
| Voices | Generates 20,000 spoken examples per phrase, plus 20,000 near-misses | about 3 h (about 4 min per 2,000) |
| Background audio | Streams about 660 hours of speech, meetings, parliament, music and noise and turns it into features | 2–4 h |
| Features | Mixes the examples into noisy rooms and computes features | 15–30 min |
| Train | Trains one small classifier per phrase | 10–20 min |
| Evaluate | Measures recall and false accepts per hour and calibrates sensitivity | 5 min |
| Export | Writes the ONNX models and `models.json`, and downloads a zip | 1 min |

In [ ]:
#@title Connect Google Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess
RUN = 'uno-v1'  #@param {type:"string"}
WORK = f'/content/drive/MyDrive/wakeword/{RUN}'
os.makedirs(WORK, exist_ok=True)
os.environ['WORK'] = WORK
print('Saving to', WORK)
gpu = shutil.which('nvidia-smi') and subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout
print(gpu or 'No GPU: switch the runtime to T4 GPU for training, or set features_use_cuda: false in the configuration to run on CPU.')

In [ ]:
#@title Install dependencies
!pip install -q piper-tts==1.8.0 onnx
# Parler-TTS for the "parler" voice group (configs/uno.yaml); it pins its own transformers version.
!pip install -q git+https://github.com/huggingface/parler-tts.git
# Parler's audio library pins protobuf below 5, but the ONNX export needs 6.31+; Parler works with it.
!pip install -q "protobuf>=6.31.1"
# piper-tts pulls in CPU onnxruntime; swap in the GPU build (same Python API).
!pip uninstall -y -q onnxruntime onnxruntime-gpu && pip install -q onnxruntime-gpu
import onnxruntime
print('onnxruntime', onnxruntime.__version__, onnxruntime.get_available_providers())

In [ ]:
#@title Write the training pipeline code (generated from the repo's training/ folder; do not edit here)
import os
FILES = {
'eval/__init__.py': '',
'eval/detector.py': '"""Python mirror of WakeWord.Core.Detection.WakeWordDetector.\n\nSection 3 metrics must be measured with the same fire rules that ship, so this module and the C#\ndetector both run testdata/golden/detector_cases.json. Change one, change the other.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nFRAME_SAMPLES = 1280\nSAMPLE_RATE = 16000\n\n\n@dataclass(frozen=True)\nclass AdaptiveThreshold:\n    points: tuple[tuple[float, float], ...] = ()\n    min_threshold: float = 0.2\n    max_threshold: float = 0.95\n\n    def threshold_for(self, base: float, noise_floor_dbfs: float) -> float:\n        offset = 0.0\n        pts = self.points\n        if pts:\n            if noise_floor_dbfs <= pts[0][0]:\n                offset = pts[0][1]\n            elif noise_floor_dbfs >= pts[-1][0]:\n                offset = pts[-1][1]\n            else:\n                for (lo_db, lo_off), (hi_db, hi_off) in zip(pts, pts[1:]):\n                    if noise_floor_dbfs <= hi_db:\n                        t = (noise_floor_dbfs - lo_db) / (hi_db - lo_db)\n                        offset = lo_off + t * (hi_off - lo_off)\n                        break\n        return min(max(base + offset, self.min_threshold), self.max_threshold)\n\n\n@dataclass(frozen=True)\nclass DetectorOptions:\n    base_threshold: float = 0.5\n    consecutive_frames: int = 3\n    refractory_seconds: float = 1.5\n    adaptive: AdaptiveThreshold = field(default_factory=AdaptiveThreshold)\n\n    @staticmethod\n    def from_json(o: dict) -> "DetectorOptions":\n        a = o.get("adaptive", {})\n        return DetectorOptions(\n            base_threshold=o["base_threshold"],\n            consecutive_frames=o["consecutive_frames"],\n            refractory_seconds=o["refractory_seconds"],\n            adaptive=AdaptiveThreshold(\n                points=tuple((float(p[0]), float(p[1])) for p in a.get("points", [])),\n                min_threshold=a.get("min_threshold", 0.2),\n                max_threshold=a.get("max_threshold", 0.95),\n            ),\n        )\n\n\nclass Detector:\n    def __init__(self, options: DetectorOptions, sample_rate: int = SAMPLE_RATE):\n        if options.consecutive_frames < 1:\n            raise ValueError("consecutive_frames must be at least 1")\n        self._o = options\n        # Same rounding as C# Math.Round (banker\'s rounding); Python\'s round() matches.\n        self._refractory_samples = round(options.refractory_seconds * sample_rate)\n        self._run = 0\n        self._last_fire: int | None = None\n\n    def process(self, score: float, noise_floor_dbfs: float, frame_end_sample: int) -> bool:\n        threshold = self._o.adaptive.threshold_for(self._o.base_threshold, noise_floor_dbfs)\n        if self._last_fire is not None and frame_end_sample - self._last_fire < self._refractory_samples:\n            self._run = 0\n            return False\n        if score < threshold:\n            self._run = 0\n            return False\n        self._run += 1\n        if self._run < self._o.consecutive_frames:\n            return False\n        self._run = 0\n        self._last_fire = frame_end_sample\n        return True\n\n    def reset(self) -> None:\n        self._run = 0\n\n\ndef fire_frames(\n    scores: Sequence[float],\n    noise_floor_dbfs: float | Sequence[float],\n    options: DetectorOptions,\n    frame_samples: int = FRAME_SAMPLES,\n    sample_rate: int = SAMPLE_RATE,\n) -> list[int]:\n    """Indices of frames that fire, for a continuous score stream where frame i ends at (i + 1) * frame_samples."""\n    detector = Detector(options, sample_rate)\n    fires = []\n    for i, score in enumerate(scores):\n        floor = noise_floor_dbfs if isinstance(noise_floor_dbfs, (int, float)) else noise_floor_dbfs[i]\n        if detector.process(score, floor, (i + 1) * frame_samples):\n            fires.append(i)\n    return fires\n\n\ndef load_golden(path: Path) -> dict:\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef golden_cases(path: Path) -> Iterable[tuple[str, list[int], list[int]]]:\n    """Yields (name, expected, actual) for each golden case."""\n    golden = load_golden(path)\n    for case in golden["cases"]:\n        actual = fire_frames(\n            case["scores"],\n            case["noise_floor_dbfs"],\n            DetectorOptions.from_json(case["options"]),\n            golden["frame_samples"],\n            golden["sample_rate"],\n        )\n        yield case["name"], case["expected_fire_frames"], actual\n',
'eval/metrics.py': '"""Section 3 acceptance metrics.\n\nA band passes only if the 95% upper confidence bound on false accepts per hour is at or under target,\nnot the raw average, so a lucky 20-hour run cannot pass a 0.2/hour target.\n\nUsage:\n    python -m eval.metrics results.json\n\nresults.json:\n    {"quiet":    {"negative_hours": 100, "false_accepts": 9,  "positives_total": 300, "positives_detected": 293},\n     "moderate": {...}, "loud": {...}}\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport sys\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass BandTarget:\n    max_fa_per_hour: float\n    max_miss_rate: float\n    min_negative_hours: float\n\n\nTARGETS: dict[str, BandTarget] = {\n    "quiet": BandTarget(max_fa_per_hour=0.2, max_miss_rate=0.03, min_negative_hours=100),\n    "moderate": BandTarget(max_fa_per_hour=0.5, max_miss_rate=0.08, min_negative_hours=50),\n    "loud": BandTarget(max_fa_per_hour=1.0, max_miss_rate=0.20, min_negative_hours=30),\n}\n\n\ndef poisson_cdf(k: int, lam: float) -> float:\n    if lam == 0:\n        return 1.0\n    log_lam = math.log(lam)\n    return sum(math.exp(i * log_lam - lam - math.lgamma(i + 1)) for i in range(k + 1))\n\n\ndef poisson_upper_bound(k: int, confidence: float = 0.95) -> float:\n    """Exact one-sided upper confidence bound on a Poisson mean after observing k events."""\n    if k < 0:\n        raise ValueError("k must be non-negative")\n    alpha = 1 - confidence\n    lo, hi = 0.0, k + 10 * math.sqrt(k + 1) + 10\n    for _ in range(200):\n        mid = (lo + hi) / 2\n        if poisson_cdf(k, mid) > alpha:\n            lo = mid\n        else:\n            hi = mid\n    return (lo + hi) / 2\n\n\ndef max_false_accepts_to_pass(target_per_hour: float, hours: float, confidence: float = 0.95) -> int:\n    """Largest observed false-accept count that still passes; -1 if the test set is too small to pass at all."""\n    k = -1\n    while poisson_upper_bound(k + 1, confidence) / hours <= target_per_hour:\n        k += 1\n    return k\n\n\n@dataclass\nclass BandResult:\n    name: str\n    negative_hours: float\n    false_accepts: int\n    positives_total: int\n    positives_detected: int\n    fa_per_hour: float = 0.0\n    fa_upper_per_hour: float = 0.0\n    miss_rate: float = 0.0\n    failures: list[str] = field(default_factory=list)\n\n    @property\n    def passed(self) -> bool:\n        return not self.failures\n\n\ndef evaluate_band(\n    name: str,\n    negative_hours: float,\n    false_accepts: int,\n    positives_total: int,\n    positives_detected: int,\n    target: BandTarget | None = None,\n) -> BandResult:\n    target = target or TARGETS[name]\n    r = BandResult(name, negative_hours, false_accepts, positives_total, positives_detected)\n    r.fa_per_hour = false_accepts / negative_hours\n    r.fa_upper_per_hour = poisson_upper_bound(false_accepts) / negative_hours\n    r.miss_rate = 1 - positives_detected / positives_total if positives_total else 1.0\n\n    if negative_hours < target.min_negative_hours:\n        r.failures.append(f"only {negative_hours:g} h of negatives; need {target.min_negative_hours:g} h")\n    if r.fa_upper_per_hour > target.max_fa_per_hour:\n        r.failures.append(f"false accepts {r.fa_upper_per_hour:.3f}/h (95% bound) > {target.max_fa_per_hour}/h")\n    if r.miss_rate > target.max_miss_rate:\n        r.failures.append(f"miss rate {r.miss_rate:.1%} > {target.max_miss_rate:.0%}")\n    return r\n\n\ndef main(argv: list[str]) -> int:\n    if len(argv) != 1:\n        print(__doc__)\n        return 2\n    results = json.loads(Path(argv[0]).read_text(encoding="utf-8"))\n    ok = True\n    print(f"{\'band\':<9} {\'hours\':>6} {\'FA\':>4} {\'FA/h\':>6} {\'FA/h 95%\':>9} {\'miss\':>6}  result")\n    for name in TARGETS:\n        if name not in results:\n            print(f"{name:<9} missing")\n            ok = False\n            continue\n        r = evaluate_band(name, **results[name])\n        ok &= r.passed\n        verdict = "PASS" if r.passed else "FAIL: " + "; ".join(r.failures)\n        print(f"{name:<9} {r.negative_hours:>6g} {r.false_accepts:>4} {r.fa_per_hour:>6.3f} {r.fa_upper_per_hour:>9.3f} {r.miss_rate:>6.1%}  {verdict}")\n    return 0 if ok else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n',
'wakeword_train/__init__.py': '"""Commercial-clean wake word training: Piper positives, openly licensed negatives, openWakeWord features.\n\nRun on Colab via training/colab/train_wake_words.ipynb, or locally with\n    python -m wakeword_train.pipeline --config configs/uno.yaml --work <dir>\n"""\n\nSAMPLE_RATE = 16000\nFRAME_SAMPLES = 1280  # 80 ms; one embedding per frame\nCLASSIFIER_FRAMES = 16  # embeddings per classifier input (1.28 s)\nEMBEDDING_DIM = 96\n',
'wakeword_train/augment.py': '"""Training-time augmentation (plan 5.4): rooms, noise, gain, microphone band-limiting, speed."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.signal import butter, fftconvolve, resample, sosfilt\n\nfrom . import SAMPLE_RATE\nfrom .config import AugmentConfig\n\n\nclass NoiseBank:\n    """Concatenated int16 noise recordings; random segments are drawn for mixing."""\n\n    def __init__(self, audio: np.ndarray):\n        if audio.size < SAMPLE_RATE:\n            raise ValueError("noise bank needs at least 1 s of audio")\n        self.audio = audio.astype(np.int16)\n\n    def segment(self, n: int, rng: np.random.Generator) -> np.ndarray:\n        if n >= self.audio.size:\n            reps = int(np.ceil(n / self.audio.size))\n            return np.tile(self.audio, reps)[:n].astype(np.float32) / 32768.0\n        start = int(rng.integers(0, self.audio.size - n))\n        return self.audio[start:start + n].astype(np.float32) / 32768.0\n\n    def save(self, path: Path) -> None:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        np.save(path, self.audio)\n\n    @staticmethod\n    def load(path: Path) -> "NoiseBank":\n        return NoiseBank(np.load(path))\n\n\ndef colored_noise(seconds: float, rng: np.random.Generator) -> np.ndarray:\n    """White, pink and brown noise, int16; licence-free filler for the noise bank."""\n    n = int(seconds * SAMPLE_RATE)\n    parts = []\n    for exponent in (0.0, 1.0, 2.0):\n        spectrum = rng.normal(size=n // 2 + 1) + 1j * rng.normal(size=n // 2 + 1)\n        f = np.arange(n // 2 + 1, dtype=np.float64)\n        f[0] = 1.0\n        x = np.fft.irfft(spectrum / f ** (exponent / 2), n)\n        parts.append(x / (np.abs(x).max() + 1e-9) * 0.3)\n    return (np.concatenate(parts) * 32767).astype(np.int16)\n\n\ndef _rms(x: np.ndarray) -> float:\n    return float(np.sqrt(np.mean(x.astype(np.float64) ** 2) + 1e-12))\n\n\nclass Augmenter:\n    def __init__(self, cfg: AugmentConfig, noise: NoiseBank, rirs: list[np.ndarray], window_samples: int, seed: int):\n        self.cfg = cfg\n        self.noise = noise\n        self.rirs = rirs\n        self.window = window_samples\n        self.rng = np.random.default_rng(seed)\n\n    def __call__(self, clip: np.ndarray, end_offset: int | None = None) -> np.ndarray:\n        """Places a clip in a window_samples window so it ends end_offset samples before the window end,\n        then augments. Returns int16 [window_samples]."""\n        rng, cfg = self.rng, self.cfg\n        x = clip.astype(np.float32) / 32768.0\n\n        if rng.random() < cfg.speed_prob:\n            rate = rng.uniform(*cfg.speed_range)\n            x = resample(x, max(1, int(len(x) / rate))).astype(np.float32)\n\n        if self.rirs and rng.random() < cfg.rir_prob:\n            rir = self.rirs[int(rng.integers(len(self.rirs)))]\n            wet = fftconvolve(x, rir)[: len(x) + SAMPLE_RATE // 4]\n            x = (wet * (np.abs(x).max() / (np.abs(wet).max() + 1e-9))).astype(np.float32)\n\n        if end_offset is None:\n            end_offset = int(rng.uniform(*cfg.trailing_seconds) * SAMPLE_RATE)\n        x = x[-(self.window - end_offset):] if len(x) > self.window - end_offset else x\n        out = np.zeros(self.window, dtype=np.float32)\n        end = self.window - end_offset\n        out[end - len(x):end] = x\n        speech_rms = _rms(x)\n\n        if rng.random() < cfg.noise_prob:\n            snr = rng.uniform(*cfg.snr_db)\n            noise = self.noise.segment(self.window, rng)\n            noise *= speech_rms / (_rms(noise) * 10 ** (snr / 20))\n            out += noise\n        out += rng.normal(scale=1e-4, size=self.window).astype(np.float32)  # never exact digital silence\n\n        if rng.random() < cfg.band_limit_prob:\n            low = rng.uniform(80, 400)\n            high = rng.uniform(3400, 7600)\n            out = sosfilt(butter(4, [low, high], btype="bandpass", fs=SAMPLE_RATE, output="sos"), out).astype(np.float32)\n\n        out *= 10 ** (rng.uniform(*cfg.gain_db) / 20)\n        peak = np.abs(out).max()\n        if peak > 0.99 and rng.random() < 0.7:\n            out *= 0.99 / peak  # mostly avoid clipping; keep some clipped examples\n        return (np.clip(out, -1, 1) * 32767).astype(np.int16)\n',
'wakeword_train/config.py': '"""Typed run configuration, loaded from YAML."""\n\nfrom __future__ import annotations\n\nimport dataclasses\nimport typing\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport yaml\n\n\n@dataclass\nclass Licensed:\n    license: str\n    license_url: str\n\n\n@dataclass\nclass PhraseConfig:\n    id: str  # file-safe id, e.g. "hey_uno"\n    display: str  # what users say, e.g. "Hey UNO"\n    tts_texts: list[str]  # spellings fed to TTS so it pronounces the phrase correctly\n    adversarial_texts: list[str]  # near-misses that must NOT fire\n    transcript_variants: list[str] = field(default_factory=list)  # Deepgram mis-hearings to strip in the app\n\n\n@dataclass\nclass VoiceConfig(Licensed):\n    name: str\n    model_url: str\n    config_url: str\n\n\n@dataclass\nclass VoiceGroup:\n    """More voices on top of tts.voices, with clips of their own (plan 5.2: every commercially licensed voice).\n\n    A run that already has clips generates only the groups it lacks, so voices can be added without redoing the rest.\n    """\n\n    name: str\n    engine: str  # "piper": voices are Piper models; "parler": voices[0] is a Parler-TTS model (Hugging Face id)\n    voices: list[VoiceConfig]\n    clips_per_phrase: int  # positives, and as many near-misses, per phrase\n    # Which speakers are held out for validation: "speakers" (every nth, like tts.voices), "all" (the whole group,\n    # to measure voices never trained on) or "none".\n    holdout: str = "speakers"\n    speakers: int = 0  # parler: how many described voices to draw from\n    enabled: bool = True  # off: kept in the config (e.g. awaiting a licence decision) but not used\n\n\n@dataclass\nclass TtsConfig:\n    voices: list[VoiceConfig]\n    positives_per_phrase: int = 20000\n    adversarial_per_phrase: int = 20000\n    length_scale: tuple[float, float] = (0.75, 1.35)\n    noise_scale: tuple[float, float] = (0.4, 1.0)\n    noise_w_scale: tuple[float, float] = (0.5, 1.2)\n    holdout_every_nth_speaker: int = 10  # speaker_id % n == 0 is held out for validation\n    use_cuda: bool = False\n    part_size: int = 2000  # clips per resumable part\n    workers: int = 1  # parallel synthesis processes; parts are the unit of work\n    groups: list[VoiceGroup] = field(default_factory=list)\n\n    @property\n    def active_groups(self) -> list[tuple[int, "VoiceGroup"]]:\n        """(group number, group) for enabled groups; numbers are positions in the config, starting at 1."""\n        return [(g, grp) for g, grp in enumerate(self.groups, start=1) if grp.enabled]\n\n    def voice(self, ref: int) -> VoiceConfig:\n        """The voice behind a clip\'s speaker id // 100000 (group * 10 + voice index)."""\n        g, v = divmod(ref, 10)\n        return self.voices[v] if g == 0 else self.groups[g - 1].voices[v]\n\n\n@dataclass\nclass NegativeSource(Licensed):\n    name: str\n    # "tar": each url is a .tar/.tar.gz of audio files; "files": each url is one audio file; "embeddings": urls[0] is\n    # a folder of .npy features already computed elsewhere (python -m wakeword_train.test_package --export-train)\n    kind: str\n    urls: list[str]\n    split: str  # "train" or "val"\n    max_hours: float\n    include: list[str] = field(default_factory=list)  # member path substrings to keep; empty keeps all audio\n    noise_bank: list[str] = field(default_factory=list)  # member path substrings also added to the augmentation noise bank\n    # Fetch each archive to disk before reading it (deleted when done): for servers that reset a stream left\n    # idle while long recordings are processed, which would otherwise mean re-streaming from the start.\n    download_first: bool = False\n    source_url: str = ""\n    release_id: str = ""  # for license Internal-Consent: reference to the participants\' consent\n\n\n@dataclass\nclass AugmentConfig:\n    copies_per_clip: int = 2\n    snr_db: tuple[float, float] = (0.0, 25.0)\n    noise_prob: float = 0.9\n    rir_prob: float = 0.5\n    gain_db: tuple[float, float] = (-12.0, 6.0)\n    band_limit_prob: float = 0.3\n    speed_prob: float = 0.3\n    speed_range: tuple[float, float] = (0.9, 1.1)\n    trailing_seconds: tuple[float, float] = (0.0, 0.4)  # audio after the phrase ends, so N consecutive frames can fire\n    noise_bank_hours: float = 6.0\n\n\n@dataclass\nclass TrainConfig:\n    steps: int = 30000\n    batch_positive: int = 256\n    batch_adversarial: int = 256\n    batch_negative: int = 512\n    layer_size: int = 128\n    n_blocks: int = 1\n    max_negative_weight: float = 1000.0\n    learning_rates: tuple[float, ...] = (1e-4, 1e-5, 1e-6)\n    phase_fractions: tuple[float, ...] = (1.0, 0.1, 0.1)\n    eval_points: int = 20\n    average_top_k: int = 5\n    seed: int = 1234\n    max_negative_hours: float = 0.0  # 0 = use all; caps RAM when the shared store is large\n\n\n@dataclass\nclass EvalConfig:\n    consecutive_frames: int = 3\n    refractory_seconds: float = 1.5\n    # Sensitivity 0..1 maps (log scale) onto these false accepts per hour on the validation set.\n    fa_per_hour_at_sensitivity_0: float = 0.05\n    fa_per_hour_at_sensitivity_1: float = 5.0\n    default_sensitivity: float = 0.5\n    val_positive_snr_db: tuple[float, float] = (5.0, 20.0)\n\n\n@dataclass\nclass TestConfig:\n    """Your own recordings with no wake word in them, such as team meetings: the frozen test set (plan 5.3).\n\n    Only ever measured on, never trained or calibrated on, so their false-accept rate stays an honest estimate.\n    """\n\n    folder: str = ""  # audio or video files, searched recursively (Meet and Teams .mp4 work); empty = no test step\n    license: str = "Internal-Consent"  # participants agreed to training use (training/data/manifest.py)\n    license_url: str = ""  # where that consent is recorded\n    release_id: str = ""  # reference to the consent or its approval\n\n\n@dataclass\nclass Config:\n    run_name: str\n    phrases: list[PhraseConfig]\n    tts: TtsConfig\n    negatives: list[NegativeSource]\n    rir: Licensed\n    rir_url: str\n    feature_models: Licensed\n    melspectrogram_url: str\n    embedding_url: str\n    augment: AugmentConfig = field(default_factory=AugmentConfig)\n    train: TrainConfig = field(default_factory=TrainConfig)\n    eval: EvalConfig = field(default_factory=EvalConfig)\n    test: TestConfig = field(default_factory=TestConfig)\n    window_samples: int = 32000  # 2.0 s of audio -> exactly 16 embeddings\n    # Folder holding assets/, negatives/ and noise/ shared by many runs (Wake Word Studio); empty = the work folder.\n    shared_dir: str = ""\n    cmudict_url: str = "https://raw.githubusercontent.com/cmusphinx/cmudict/master/cmudict.dict"\n    feature_threads: int = 0  # 0 = all cores\n    features_use_cuda: bool = False  # needs onnxruntime-gpu (the Colab notebook installs it)\n\n    def phrase(self, phrase_id: str) -> PhraseConfig:\n        return next(p for p in self.phrases if p.id == phrase_id)\n\n\ndef _build(tp, value):\n    origin = typing.get_origin(tp)\n    if dataclasses.is_dataclass(tp):\n        hints = typing.get_type_hints(tp)\n        known = {f.name for f in dataclasses.fields(tp)}\n        unknown = set(value) - known\n        if unknown:\n            raise ValueError(f"{tp.__name__}: unknown keys {sorted(unknown)}")\n        return tp(**{k: _build(hints[k], v) for k, v in value.items()})\n    if origin is list:\n        (item,) = typing.get_args(tp)\n        return [_build(item, v) for v in value]\n    if origin is tuple:\n        return tuple(value)\n    return value\n\n\ndef load_config(path: str | Path, overrides: dict | None = None) -> Config:\n    data = yaml.safe_load(Path(path).read_text(encoding="utf-8"))\n    for dotted, v in (overrides or {}).items():\n        node = data\n        *parents, leaf = dotted.split(".")\n        for p in parents:\n            node = node.setdefault(p, {})\n        node[leaf] = v\n    return _build(Config, data)\n',
'wakeword_train/evaluate.py': '"""Streaming evaluation with the shipped fire rules, DET sweep, and sensitivity calibration."""\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\nimport torch\n\nfrom . import CLASSIFIER_FRAMES, FRAME_SAMPLES, SAMPLE_RATE\nfrom .config import EvalConfig\nfrom .store import FRAMES_PER_HOUR\nfrom eval.metrics import poisson_upper_bound\n\n\ndef count_fires(scores: np.ndarray, threshold: float, consecutive: int, refractory_seconds: float) -> np.ndarray:\n    """Frame indices that fire, for a constant threshold.\n\n    Identical to training/eval/detector.py (and the C# WakeWordDetector) with no adaptive offset:\n    N consecutive frames at or above threshold fire; frames inside the refractory period after a fire\n    are ignored and break runs. Vectorised over runs so threshold sweeps over ~1M frames stay fast.\n    """\n    refractory_samples = round(refractory_seconds * SAMPLE_RATE)\n    blocked_frames = -(-refractory_samples // FRAME_SAMPLES)  # frames after a fire that are still refractory\n    above = np.concatenate([[False], scores >= threshold, [False]])\n    edges = np.diff(above.astype(np.int8))\n    starts, ends = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)\n    fires: list[int] = []\n    last = None\n    for s, e in zip(starts.tolist(), ends.tolist()):\n        j = s\n        while True:\n            if last is not None:\n                j = max(j, last + max(blocked_frames, 1))\n            f = j + consecutive - 1\n            if f >= e:\n                break\n            fires.append(f)\n            last = f\n            j = f + 1\n    return np.asarray(fires, dtype=np.int64)\n\n\n@torch.no_grad()\ndef window_scores(model: torch.nn.Module, windows: np.ndarray | torch.Tensor, device: str, batch: int = 65536) -> np.ndarray:\n    out = []\n    for i in range(0, len(windows), batch):\n        x = torch.as_tensor(np.asarray(windows[i:i + batch]) if not torch.is_tensor(windows) else windows[i:i + batch])\n        out.append(model(x.to(device, torch.float32)).squeeze(1).float().cpu().numpy())\n    return np.concatenate(out) if out else np.empty(0, np.float32)\n\n\n@torch.no_grad()\ndef stream_scores(model: torch.nn.Module, stream: np.ndarray | torch.Tensor, device: str, batch: int = 65536) -> np.ndarray:\n    """Score for every position of a [T, 96] embedding stream: score k uses embeddings k..k+15."""\n    s = torch.as_tensor(stream) if not torch.is_tensor(stream) else stream\n    s = s.to(device)\n    n = s.shape[0] - CLASSIFIER_FRAMES + 1\n    out = []\n    for i in range(0, max(n, 0), batch):\n        idx = torch.arange(i, min(i + batch, n), device=device)[:, None] + torch.arange(CLASSIFIER_FRAMES, device=device)\n        out.append(model(s[idx].float()).squeeze(1).float().cpu().numpy())\n    return np.concatenate(out) if out else np.empty(0, np.float32)\n\n\n@dataclass\nclass DetPoint:\n    threshold: float\n    recall: float\n    fa_per_hour: float\n    false_accepts: int\n\n\n@dataclass\nclass PhraseEvaluation:\n    phrase: str\n    negative_hours: float\n    positives: int\n    det: list[DetPoint]\n    sensitivity_table: list[tuple[float, float]]  # (sensitivity, threshold)\n    default_sensitivity: float\n    default_threshold: float\n    default_recall: float\n    default_fa_per_hour: float\n\n    def to_json(self) -> dict:\n        d = asdict(self)\n        d["sensitivity_table"] = [list(p) for p in self.sensitivity_table]\n        return d\n\n\n# Down to the detectors\' min_threshold (0.01): real voices score lower than the synthetic validation clips.\nTHRESHOLDS = np.unique(np.round(np.concatenate([\n    np.arange(0.01, 0.02, 0.0025), np.arange(0.02, 0.99, 0.01), np.arange(0.99, 0.9995, 0.001)]), 4))\n\n\ndef sweep(\n    positive_scores: list[np.ndarray],\n    negative_scores: np.ndarray,\n    cfg: EvalConfig,\n    thresholds: np.ndarray = THRESHOLDS,\n) -> list[DetPoint]:\n    hours = len(negative_scores) / FRAMES_PER_HOUR\n    points = []\n    for t in thresholds:\n        detected = sum(\n            1 for s in positive_scores if len(count_fires(s, t, cfg.consecutive_frames, cfg.refractory_seconds)) > 0\n        )\n        fa = len(count_fires(negative_scores, t, cfg.consecutive_frames, cfg.refractory_seconds))\n        points.append(DetPoint(float(t), detected / max(1, len(positive_scores)), fa / hours if hours else math.inf, fa))\n    return points\n\n\ndef target_fa_per_hour(sensitivity: float, cfg: EvalConfig) -> float:\n    lo, hi = math.log10(cfg.fa_per_hour_at_sensitivity_0), math.log10(cfg.fa_per_hour_at_sensitivity_1)\n    return 10 ** (lo + sensitivity * (hi - lo))\n\n\ndef calibrate(det: list[DetPoint], cfg: EvalConfig, negative_hours: float) -> list[tuple[float, float]]:\n    """Sensitivity 0..1 (step 0.1) -> the lowest threshold whose false-accept rate is within that sensitivity\'s\n    target at 95% confidence (Poisson upper bound, as in Section 3).\n\n    Like Picovoice\'s sensitivity: higher means fewer misses and more false accepts. A target the validation\n    audio is too short to prove gets the strictest threshold, never a guess.\n    """\n    by_threshold = sorted(det, key=lambda p: p.threshold)\n    table = []\n    for s in np.round(np.arange(0.0, 1.0001, 0.1), 1):\n        target = target_fa_per_hour(float(s), cfg)\n        ok = [p.threshold for p in by_threshold if negative_hours > 0\n              and poisson_upper_bound(p.false_accepts) / negative_hours <= target]\n        table.append((float(s), ok[0] if ok else by_threshold[-1].threshold))\n    # Monotonic: more sensitive never means a higher threshold.\n    for i in range(1, len(table)):\n        table[i] = (table[i][0], min(table[i][1], table[i - 1][1]))\n    return table\n\n\ndef threshold_for(sensitivity: float, table: list[tuple[float, float]]) -> float:\n    """Linear interpolation in the calibration table; the apps do the same."""\n    s = min(max(sensitivity, 0.0), 1.0)\n    for (s0, t0), (s1, t1) in zip(table, table[1:]):\n        if s <= s1:\n            return t0 + (t1 - t0) * (s - s0) / (s1 - s0)\n    return table[-1][1]\n\n\ndef evaluate_phrase(\n    phrase: str,\n    positive_scores: list[np.ndarray],\n    negative_scores: np.ndarray,\n    cfg: EvalConfig,\n) -> PhraseEvaluation:\n    det = sweep(positive_scores, negative_scores, cfg)\n    table = calibrate(det, cfg, len(negative_scores) / FRAMES_PER_HOUR)\n    t = threshold_for(cfg.default_sensitivity, table)\n    nearest = min(det, key=lambda p: abs(p.threshold - t))\n    return PhraseEvaluation(\n        phrase=phrase,\n        negative_hours=len(negative_scores) / FRAMES_PER_HOUR,\n        positives=len(positive_scores),\n        det=det,\n        sensitivity_table=table,\n        default_sensitivity=cfg.default_sensitivity,\n        default_threshold=round(t, 4),\n        default_recall=nearest.recall,\n        default_fa_per_hour=nearest.fa_per_hour,\n    )\n',
'wakeword_train/export.py': '"""ONNX export, parity checks, golden vectors and the model package the apps load."""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport shutil\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport onnxruntime as ort\nimport torch\n\nfrom . import CLASSIFIER_FRAMES, EMBEDDING_DIM, FRAME_SAMPLES, SAMPLE_RATE\nfrom .config import Config\nfrom .features import MEL_STEP, MEL_WINDOW, FeatureExtractor\n\nPACKAGE_FORMAT = "wakeword-models/1"\n\n\ndef export_onnx(model: torch.nn.Module, path: Path) -> None:\n    model = model.cpu().eval()\n    dummy = torch.zeros(1, CLASSIFIER_FRAMES, EMBEDDING_DIM)\n    torch.onnx.export(\n        model, (dummy,), str(path),\n        input_names=["input"], output_names=["score"],\n        dynamic_axes={"input": {0: "batch"}, "score": {0: "batch"}},\n        opset_version=17, dynamo=False,\n    )\n\n\ndef check_parity(model: torch.nn.Module, path: Path, windows: np.ndarray, tolerance: float = 1e-4) -> float:\n    """Max absolute difference between PyTorch and ONNX Runtime scores on real windows."""\n    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])\n    x = windows.astype(np.float32)\n    onnx_scores = session.run(None, {"input": x})[0]\n    with torch.no_grad():\n        torch_scores = model.cpu().eval()(torch.from_numpy(x)).numpy()\n    diff = float(np.abs(onnx_scores - torch_scores).max())\n    if diff > tolerance:\n        raise RuntimeError(f"ONNX parity failed for {path.name}: max diff {diff}")\n    return diff\n\n\ndef golden_vectors(fx: FeatureExtractor, classifiers: dict[str, Path], audio: np.ndarray) -> dict:\n    """Runs the exact runtime streaming pipeline over audio and records every intermediate the apps must match."""\n    sessions = {k: ort.InferenceSession(str(p), providers=["CPUExecutionProvider"]) for k, p in classifiers.items()}\n    stream = fx.streaming()\n    chunks = []\n    n = len(audio) // FRAME_SAMPLES\n    for i in range(n):\n        produced = stream.push(audio[i * FRAME_SAMPLES:(i + 1) * FRAME_SAMPLES])\n        x = stream.classifier_input()\n        entry = {"chunk": i, "embedding_produced": produced}\n        if produced:\n            entry["embedding_first8"] = [round(float(v), 5) for v in stream.embeddings[-1][:8]]\n        if x is not None:\n            entry["scores"] = {k: round(float(s.run(None, {"input": x})[0][0, 0]), 6) for k, s in sessions.items()}\n        chunks.append(entry)\n    return {\n        "description": "Runtime feature + classifier golden vectors. Feed audio in 1280-sample chunks; "\n                       "C# and web implementations must match scores within 1e-3.",\n        "sample_rate": SAMPLE_RATE,\n        "frame_samples": FRAME_SAMPLES,\n        "audio_int16": audio[: n * FRAME_SAMPLES].astype(int).tolist(),\n        "chunks": chunks,\n    }\n\n\n# The pipeline\'s "val" split is the plan\'s dev split (5.3): it picks thresholds. "test" is kept for real recordings.\nMANIFEST_SPLIT = {"train": "train", "val": "dev"}\n\n\ndef write_dataset_manifest(\n    cfg: Config,\n    hours: dict[str, float],\n    path: Path,\n    tts_seconds: dict[tuple[int, str, str], float],\n    rir_seconds: float,\n    test_hours: float = 0.0,\n) -> None:\n    """Corpus-level rows that pass training/data/manifest.py: one per voice and use, negative source and RIR set.\n\n    tts_seconds maps (voice index, kind, split) to seconds of synthetic clips; kind is "positive" or\n    "hard_negative" (near-misses), split "train" or "dev" (held-out speakers). Sources that contributed\n    no audio are left out.\n    """\n    today = datetime.now(timezone.utc).date().isoformat()\n    rows = []\n\n    def row(path, kind, split, source, url, licensed, seconds, speaker=""):\n        rows.append({"path": path, "kind": kind, "split": split, "source": source, "url": url,\n                     "license": licensed.license, "license_url": licensed.license_url, "retrieved": today,\n                     "duration_seconds": round(seconds, 1), "speaker_id": speaker, "release_id": ""})\n\n    for (ref, kind, split), seconds in sorted(tts_seconds.items()):\n        v = cfg.tts.voice(ref)\n        if seconds > 0:\n            # Synthetic speakers are tracked per voice; held-out ones get their own id so they never share a split.\n            speaker = (v.name + ("/held-out" if split == "dev" else "")) if kind == "positive" else ""\n            engine = "Parler-TTS" if ref >= 10 and cfg.tts.groups[ref // 10 - 1].engine == "parler" else "Piper voice"\n            row(f"tts/{v.name}/{kind}/{split}", kind, split, f"{engine} {v.name}", v.model_url, v, seconds, speaker)\n    for src in cfg.negatives:\n        if hours.get(src.name, 0.0) > 0:\n            row(f"negatives/{src.name}", "negative", MANIFEST_SPLIT[src.split], src.name, src.source_url or src.urls[0],\n                src, hours[src.name] * 3600)\n            rows[-1]["release_id"] = src.release_id\n    if rir_seconds > 0:\n        row("rir/mit", "rir", "train", "MIT Acoustical Reverberation Scene Statistics Survey", cfg.rir_url, cfg.rir, rir_seconds)\n    if test_hours > 0:  # one aggregate row: file names of internal recordings stay out of the package\n        row("test/own-recordings", "negative", "test", "Own recordings (internal, with consent)", "internal", cfg.test,\n            test_hours * 3600)\n        rows[-1]["release_id"] = cfg.test.release_id\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\n        w.writeheader()\n        w.writerows(rows)\n\n\ndef write_package(\n    cfg: Config,\n    out_dir: Path,\n    feature_models: dict[str, Path],\n    classifiers: dict[str, Path],\n    evaluations: dict[str, dict],\n    hours: dict[str, float],\n    golden: dict,\n    notes: list[str],\n    tts_seconds: dict[tuple[int, str, str], float],\n    rir_seconds: float,\n    tests: dict[str, dict] | None = None,\n) -> Path:\n    tests = tests or {}\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for name, p in {**feature_models, **classifiers}.items():\n        shutil.copy(p, out_dir / p.name)\n\n    keywords = []\n    for phrase in cfg.phrases:\n        ev = evaluations[phrase.id]\n        keywords.append({\n            "id": phrase.id,\n            "phrase": phrase.display,\n            "model": classifiers[phrase.id].name,\n            "input": "input",\n            "output": "score",\n            "default_sensitivity": cfg.eval.default_sensitivity,\n            "sensitivity_table": ev["sensitivity_table"],\n            # Thresholds come from the calibration table, so the app\'s clamp must not cut them off.\n            "detector": {"consecutive_frames": cfg.eval.consecutive_frames, "refractory_seconds": cfg.eval.refractory_seconds,\n                         "min_threshold": 0.01, "max_threshold": 0.9999},\n            "transcript_variants": phrase.transcript_variants,\n            "validation": {\n                "threshold": ev["default_threshold"],\n                "recall": ev["default_recall"],\n                "fa_per_hour": ev["default_fa_per_hour"],\n                "negative_hours": ev["negative_hours"],\n                "positives": ev["positives"],\n            },\n        })\n        if phrase.id in tests:  # your own recordings, at the default sensitivity\n            t = tests[phrase.id]\n            d = next(r for r in t["by_sensitivity"] if r["sensitivity"] == cfg.eval.default_sensitivity)\n            keywords[-1]["test"] = {"hours": t["hours"], "false_accepts": d["false_accepts"],\n                                    "fa_per_hour": d["fa_per_hour"], "fa_per_hour_upper": d["fa_per_hour_upper"]}\n\n    manifest = {\n        "format": PACKAGE_FORMAT,\n        "run": cfg.run_name,\n        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),\n        "sample_rate": SAMPLE_RATE,\n        "frame_samples": FRAME_SAMPLES,\n        "features": {\n            "melspectrogram": feature_models["melspectrogram"].name,\n            "embedding": feature_models["embedding"].name,\n            "mel_transform": "x / 10 + 2",\n            "mel_context_samples": 480,\n            "embedding_window": MEL_WINDOW,\n            "embedding_step": MEL_STEP,\n            "classifier_frames": CLASSIFIER_FRAMES,\n        },\n        "keywords": keywords,\n        "training_hours": {k: round(v, 2) for k, v in hours.items()},\n        "notes": notes,\n    }\n    (out_dir / "models.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    (out_dir / "golden.json").write_text(json.dumps(golden), encoding="utf-8")\n    test_hours = next(iter(tests.values()))["hours"] if tests else 0.0\n    write_dataset_manifest(cfg, hours, out_dir / "DATASET_MANIFEST.csv", tts_seconds, rir_seconds, test_hours)\n    (out_dir / "METRICS.md").write_text(metrics_markdown(cfg, evaluations, hours, notes, tests), encoding="utf-8")\n\n    zip_path = out_dir.with_suffix(".zip")\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out_dir.iterdir()):\n            z.write(p, p.name)\n    return zip_path\n\n\ndef metrics_markdown(cfg: Config, evaluations: dict[str, dict], hours: dict[str, float], notes: list[str],\n                     tests: dict[str, dict] | None = None) -> str:\n    lines = [f"# Wake word models: {cfg.run_name}", ""]\n    lines += [f"- {n}" for n in notes] + [""]\n    lines += ["| Keyword | Sensitivity | Threshold | Recall (held-out synthetic voices) | False accepts / hour | Negative hours |",\n              "| --- | --- | --- | --- | --- | --- |"]\n    for p in cfg.phrases:\n        ev = evaluations[p.id]\n        lines.append(f"| {p.display} | {ev[\'default_sensitivity\']} | {ev[\'default_threshold\']:.3f} | "\n                     f"{ev[\'default_recall\']:.1%} | {ev[\'default_fa_per_hour\']:.2f} | {ev[\'negative_hours\']:.1f} |")\n    lines += ["", "## Sensitivity calibration", "", "| Sensitivity | " + " | ".join(p.display for p in cfg.phrases) + " |",\n              "| --- |" + " --- |" * len(cfg.phrases)]\n    tables = {p.id: dict((round(s, 1), t) for s, t in evaluations[p.id]["sensitivity_table"]) for p in cfg.phrases}\n    for s in np.round(np.arange(0.0, 1.0001, 0.1), 1):\n        lines.append(f"| {s:.1f} | " + " | ".join(f"{tables[p.id][float(s)]:.3f}" for p in cfg.phrases) + " |")\n    if tests:\n        first = next(iter(tests.values()))\n        lines += ["", "## Test: your own recordings (never trained or calibrated on)", "",\n                  f"{first[\'files\']} recordings, {first[\'hours\']:.1f} h. False accepts per hour, with the 95% upper bound "\n                  "in brackets; the Section 3 target is at most 0.2 (quiet) to 1.0 (loud).", "",\n                  "| Sensitivity | " + " | ".join(p.display for p in cfg.phrases) + " |", "| --- |" + " --- |" * len(cfg.phrases)]\n        for i, r in enumerate(first["by_sensitivity"]):\n            cells = [tests[p.id]["by_sensitivity"][i] for p in cfg.phrases]\n            lines.append(f"| {r[\'sensitivity\']:.1f} | " + " | ".join(\n                f"{c[\'fa_per_hour\']:.2f} ({c[\'fa_per_hour_upper\']:.2f})" for c in cells) + " |")\n    lines += ["", "## Training audio (hours)", ""] + [f"- {k}: {v:.1f}" for k, v in hours.items()]\n    return "\\n".join(lines) + "\\n"\n',
'wakeword_train/features.py': '"""openWakeWord feature extraction: melspectrogram.onnx -> embedding_model.onnx (Google speech_embedding).\n\nBatch functions are used for training. StreamingFeatures defines the exact runtime behaviour that the\nC# and web clients must reproduce (checked with golden vectors from export.py).\n"""\n\nfrom __future__ import annotations\n\nimport os\n\nimport numpy as np\nimport onnxruntime as ort\n\nfrom . import CLASSIFIER_FRAMES, EMBEDDING_DIM, FRAME_SAMPLES, SAMPLE_RATE\n\nMEL_WINDOW = 76  # mel frames per embedding\nMEL_STEP = 8  # mel frames per 1280-sample chunk\nMEL_CONTEXT = 160 * 3  # extra samples fed to the mel model per streaming chunk\nMEL_FFT, MEL_HOP = 512, 160  # the mel model rejects input under 512 samples; frames = (N - 512) // 160 + 1\nMIN_EMBED_SAMPLES = MEL_FFT + (MEL_WINDOW - 1) * MEL_HOP  # shortest audio that yields one embedding\n\n\ndef mel_transform(spec: np.ndarray) -> np.ndarray:\n    # Matches openWakeWord: brings the ONNX mel model close to the original TF implementation.\n    return spec / 10.0 + 2.0\n\n\nclass FeatureExtractor:\n    def __init__(self, melspectrogram_path: str, embedding_path: str, threads: int = 0, use_cuda: bool = False):\n        options = ort.SessionOptions()\n        n = threads or os.cpu_count() or 1\n        options.intra_op_num_threads = n\n        options.inter_op_num_threads = 1\n        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if use_cuda else ["CPUExecutionProvider"]\n        self._mel = ort.InferenceSession(melspectrogram_path, options, providers=providers)\n        self._emb = ort.InferenceSession(embedding_path, options, providers=providers)\n        self._mel_in = self._mel.get_inputs()[0].name\n        self._emb_in = self._emb.get_inputs()[0].name\n\n    def melspectrogram(self, audio: np.ndarray) -> np.ndarray:\n        """int16 audio [N] or [B, N] -> float32 mel [frames, 32] or [B, frames, 32]."""\n        if audio.dtype != np.int16:\n            raise ValueError(f"audio must be int16, got {audio.dtype}")\n        batched = audio.ndim == 2\n        x = (audio if batched else audio[None]).astype(np.float32)\n        spec = self._mel.run(None, {self._mel_in: x})[0]  # [B, 1, frames, 32]\n        spec = mel_transform(spec.reshape(x.shape[0], -1, 32))\n        return spec if batched else spec[0]\n\n    def embed_mel_windows(self, windows: np.ndarray, batch_size: int = 1024) -> np.ndarray:\n        """[B, 76, 32] mel windows -> [B, 96] embeddings."""\n        out = np.empty((windows.shape[0], EMBEDDING_DIM), dtype=np.float32)\n        for i in range(0, windows.shape[0], batch_size):\n            chunk = windows[i:i + batch_size, :, :, None].astype(np.float32)\n            out[i:i + batch_size] = self._emb.run(None, {self._emb_in: chunk})[0].reshape(chunk.shape[0], EMBEDDING_DIM)\n        return out\n\n    def embed(self, audio: np.ndarray) -> np.ndarray:\n        """int16 audio of any length -> [T, 96], one embedding per 1280 samples once 76 mel frames exist."""\n        if audio.shape[0] < MIN_EMBED_SAMPLES:\n            return np.empty((0, EMBEDDING_DIM), dtype=np.float32)\n        spec = self.melspectrogram(audio)\n        starts = range(0, spec.shape[0] - MEL_WINDOW + 1, MEL_STEP)\n        if not starts:\n            return np.empty((0, EMBEDDING_DIM), dtype=np.float32)\n        windows = np.stack([spec[s:s + MEL_WINDOW] for s in starts])\n        return self.embed_mel_windows(windows)\n\n    def embed_long(self, audio: np.ndarray, chunk_samples: int = SAMPLE_RATE * 600) -> np.ndarray:\n        """embed() for recordings hours long, in bounded memory, with an identical result.\n\n        Each chunk starts where the previous chunk\'s next embedding would start (a multiple of 1280\n        samples, so mel frames stay on the same grid), so chunking never drops or duplicates an embedding.\n        """\n        out, pos = [], 0\n        while audio.shape[0] - pos >= MIN_EMBED_SAMPLES:\n            emb = self.embed(audio[pos:pos + max(chunk_samples, MIN_EMBED_SAMPLES)])\n            out.append(emb)\n            pos += emb.shape[0] * FRAME_SAMPLES\n        return np.concatenate(out) if out else np.empty((0, EMBEDDING_DIM), dtype=np.float32)\n\n    def embed_batch(self, clips: np.ndarray, batch_size: int = 64) -> np.ndarray:\n        """int16 [B, N] equal-length clips -> [B, T, 96]."""\n        results = []\n        for i in range(0, clips.shape[0], batch_size):\n            batch = clips[i:i + batch_size]\n            try:\n                spec = self.melspectrogram(batch)\n            except Exception:  # some ORT builds reject batched mel input; fall back per clip\n                spec = np.stack([self.melspectrogram(c) for c in batch])\n            starts = list(range(0, spec.shape[1] - MEL_WINDOW + 1, MEL_STEP))\n            windows = np.stack([spec[:, s:s + MEL_WINDOW] for s in starts], axis=1)  # [b, T, 76, 32]\n            b, t = windows.shape[:2]\n            emb = self.embed_mel_windows(windows.reshape(b * t, MEL_WINDOW, 32))\n            results.append(emb.reshape(b, t, EMBEDDING_DIM))\n        return np.concatenate(results)\n\n    def streaming(self) -> "StreamingFeatures":\n        return StreamingFeatures(self)\n\n\nclass StreamingFeatures:\n    """Runtime feature pipeline, chunk by chunk, exactly as the apps run it.\n\n    For each 1280-sample chunk: run mel over the chunk plus the previous 480 samples (8 new frames; the\n    very first chunk has no context and yields 5), append to the mel buffer, and once 77 mel frames exist\n    compute one embedding from the 76 frames before the newest one. A classifier score is available once\n    16 embeddings exist.\n\n    Skipping the newest frame puts every window on the same 8-frame grid as FeatureExtractor.embed (training):\n    after chunk k the buffer holds frames 0..8k+4, and training windows end at frames 8j+75, so the window\n    ending at 8k+3 is training window j = k - 9. It costs 10 ms of latency.\n    """\n\n    def __init__(self, fx: FeatureExtractor):\n        self._fx = fx\n        self._prev = np.empty(0, dtype=np.int16)\n        self._mel = np.empty((0, 32), dtype=np.float32)\n        self.embeddings = np.empty((0, EMBEDDING_DIM), dtype=np.float32)\n\n    def push(self, chunk: np.ndarray) -> bool:\n        """Adds one 1280-sample chunk. Returns True if a new embedding was produced."""\n        if chunk.shape != (FRAME_SAMPLES,) or chunk.dtype != np.int16:\n            raise ValueError("chunks must be 1280 int16 samples")\n        audio = np.concatenate([self._prev[-MEL_CONTEXT:], chunk])\n        self._prev = audio\n        self._mel = np.concatenate([self._mel, self._fx.melspectrogram(audio)])[-(MEL_WINDOW + 1 + MEL_STEP):]\n        if self._mel.shape[0] < MEL_WINDOW + 1:\n            return False\n        emb = self._fx.embed_mel_windows(self._mel[None, -(MEL_WINDOW + 1):-1])\n        self.embeddings = np.concatenate([self.embeddings, emb])[-CLASSIFIER_FRAMES:]\n        return True\n\n    def classifier_input(self) -> np.ndarray | None:\n        return self.embeddings[None].astype(np.float32) if self.embeddings.shape[0] == CLASSIFIER_FRAMES else None\n',
'wakeword_train/model.py': '"""Classifier head over 16 x 96 embeddings (same architecture as openWakeWord\'s "dnn")."""\n\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\n\nfrom . import CLASSIFIER_FRAMES, EMBEDDING_DIM\n\n\nclass WakeWordDNN(nn.Module):\n    def __init__(self, layer_size: int = 128, n_blocks: int = 1):\n        super().__init__()\n        layers: list[nn.Module] = [\n            nn.Flatten(),\n            nn.Linear(CLASSIFIER_FRAMES * EMBEDDING_DIM, layer_size),\n            nn.LayerNorm(layer_size),\n            nn.ReLU(),\n        ]\n        for _ in range(n_blocks):\n            layers += [nn.Linear(layer_size, layer_size), nn.LayerNorm(layer_size), nn.ReLU()]\n        layers += [nn.Linear(layer_size, 1), nn.Sigmoid()]\n        self.net = nn.Sequential(*layers)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        """[B, 16, 96] -> [B, 1] score in [0, 1]."""\n        return self.net(x)\n\n\ndef average_state_dicts(states: list[dict]) -> dict:\n    return {k: sum(s[k].float() for s in states) / len(states) for k in states[0]}\n',
'wakeword_train/nearmiss.py': '"""Near-miss phrases for any wake phrase, so a phrase typed into Wake Word Studio gets hard negatives\nautomatically (as hand-written ones were for "Hey UNO").\n\nUses the CMU Pronouncing Dictionary (BSD-2-Clause, downloaded by the setup step): for each word, words\none phoneme away (substituted, dropped or added), plus the phrase\'s own fragments and common speech.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nCOMMON = [\n    "hey", "hello", "okay", "you know", "hey there", "hello there", "excuse me", "thank you",\n    "what time is it", "I don\'t know", "see you later", "come here",\n]\n\n_WORD = re.compile(r"^[a-z][a-z\']*$")\n\n\nclass Pronunciations:\n    def __init__(self, entries: dict[str, list[tuple[str, ...]]]):\n        self.entries = entries\n        self.by_phones: dict[tuple[str, ...], list[str]] = defaultdict(list)\n        for word, prons in entries.items():\n            for p in prons:\n                self.by_phones[p].append(word)\n        self.phones = sorted({ph for prons in entries.values() for p in prons for ph in p})\n\n    @staticmethod\n    def load(path: Path) -> "Pronunciations":\n        entries: dict[str, list[tuple[str, ...]]] = defaultdict(list)\n        for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():\n            line = line.split("#", 1)[0].strip()\n            if not line:\n                continue\n            word, *phones = line.split()\n            word = re.sub(r"\\(\\d+\\)$", "", word.lower())\n            if _WORD.match(word):\n                entries[word].append(tuple(re.sub(r"\\d", "", p) for p in phones))  # drop stress marks\n        return Pronunciations(dict(entries))\n\n    def neighbours(self, word: str) -> list[str]:\n        """Words whose pronunciation is one phoneme away from any pronunciation of `word`."""\n        found: set[str] = set()\n        for p in self.entries.get(word.lower(), []):\n            for i in range(len(p) + 1):\n                for ph in self.phones:\n                    found.update(self.by_phones.get(p[:i] + (ph,) + p[i:], []))  # insert\n                if i < len(p):\n                    found.update(self.by_phones.get(p[:i] + p[i + 1:], []))  # drop\n                    for ph in self.phones:\n                        if ph != p[i]:\n                            found.update(self.by_phones.get(p[:i] + (ph,) + p[i + 1:], []))  # substitute\n        found.discard(word.lower())\n        return sorted(w for w in found if len(w) >= 2)\n\n\ndef near_misses(phrase: str, pron: Pronunciations, per_word: int = 6, limit: int = 40, seed: int = 0) -> list[str]:\n    words = phrase.lower().split()\n    rng = np.random.default_rng(seed)\n    out: list[str] = []\n\n    # Fragments: each word alone, every prefix and suffix ("hey", "computer", "hey" + ...).\n    out += words if len(words) > 1 else []\n    out += [" ".join(words[:k]) for k in range(1, len(words))]\n    out += [" ".join(words[k:]) for k in range(1, len(words))]\n\n    # One word swapped for a word that sounds almost the same ("hey commuter").\n    for i, w in enumerate(words):\n        candidates = pron.neighbours(w)\n        if i == len(words) - 1:\n            # "computers", "computer\'s": a sound added at the very end is too close to the phrase itself\n            # to train as a negative without hurting recall.\n            candidates = [c for c in candidates if not c.startswith(w)]\n        if not candidates:\n            continue\n        picks = rng.choice(candidates, size=min(per_word, len(candidates)), replace=False)\n        out += [" ".join(words[:i] + [str(c)] + words[i + 1:]) for c in picks]\n\n    out += COMMON\n    seen, result = {phrase.lower()}, []\n    for text in out:\n        if text and text not in seen:\n            seen.add(text)\n            result.append(text)\n    return result[:limit]\n\n\ndef phrase_id(phrase: str) -> str:\n    return re.sub(r"[^a-z0-9]+", "_", phrase.lower()).strip("_") or "phrase"\n\n\ndef phrase_config(phrase: str, pron: Pronunciations) -> dict:\n    """A `phrases` entry for the pipeline config."""\n    words = phrase.split()\n    text = " ".join(words).lower()\n    spellings = [text, f"{text}!", f"{text}?"]\n    if len(words) > 1:\n        spellings.append(f"{words[0].lower()}, {\' \'.join(words[1:]).lower()}")\n    return {\n        "id": phrase_id(phrase),\n        "display": " ".join(words),\n        "tts_texts": spellings,\n        "adversarial_texts": near_misses(phrase, pron),\n        "transcript_variants": [text],\n    }\n',
'wakeword_train/pipeline.py': '"""Resumable training pipeline.\n\n    python -m wakeword_train.pipeline --config configs/uno.yaml --work /content/drive/MyDrive/wakeword/uno --steps all\n\nSteps (in order): setup, preview, tts, negatives, features, train, evaluate, export.\nEach step writes work/steps/<step>.done; finished steps are skipped unless --force. Long steps (tts,\nnegatives) also resume part-way, so a dropped Colab session loses minutes, not hours.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport multiprocessing\nimport os\nimport tempfile\nimport time\nimport wave\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport numpy as np\nimport yaml\n\nfrom . import SAMPLE_RATE\nfrom .augment import Augmenter, NoiseBank, colored_noise\nfrom .config import AugmentConfig, Config, load_config\nfrom .features import MIN_EMBED_SAMPLES, FeatureExtractor\nfrom .sources import download, iter_file_audio, iter_tar_audio, load_rirs\nfrom .store import EmbeddingStore, FRAMES_PER_HOUR\nfrom .tts import GROUP_SPEAKERS, ClipSet, ParlerSynth, PiperSynth, generate, generate_parler, held_out_mask, parler_descriptions\n\nSTEPS = ["setup", "preview", "tts", "negatives", "features", "train", "evaluate", "test", "export"]\nSHARED_DIRS = ("assets", "noise", "negatives")\nEVAL_WINDOW = 4 * SAMPLE_RATE  # validation positives: clip ends 1 s before the end of a 4 s window\nEVAL_TRAILING = SAMPLE_RATE\n\n\nclass Run:\n    def __init__(self, cfg: Config, work: Path, negatives_split: str | None = None):\n        self.cfg = cfg\n        self.work = work\n        self.negatives_split = negatives_split  # limits the negatives step to "train" or "val" sources\n        # Background audio, noise and downloaded models can be shared by many runs (Wake Word Studio).\n        self.shared = Path(cfg.shared_dir) if cfg.shared_dir else work\n        for d in ("steps", "clips", "features", "models", "eval", "export", "preview"):\n            (work / d).mkdir(parents=True, exist_ok=True)\n        for d in SHARED_DIRS:\n            (self.shared / d).mkdir(parents=True, exist_ok=True)\n\n    # ---- paths and shared resources\n\n    def asset(self, name: str) -> Path:\n        return self.shared / "assets" / name\n\n    def voice_paths(self, voices=None) -> list[tuple[Path, Path]]:\n        voices = self.cfg.tts.voices if voices is None else voices\n        return [(self.asset(f"voices/{v.name}.onnx"), self.asset(f"voices/{v.name}.onnx.json")) for v in voices]\n\n    def features(self) -> FeatureExtractor:\n        return FeatureExtractor(str(self.asset("melspectrogram.onnx")), str(self.asset("embedding_model.onnx")),\n                                self.cfg.feature_threads, self.cfg.features_use_cuda)\n\n    def synths(self) -> list[PiperSynth]:\n        return [PiperSynth(m, c, self.cfg.tts.use_cuda) for m, c in self.voice_paths()]\n\n    def store(self, split: str) -> EmbeddingStore:\n        return EmbeddingStore(self.shared / "negatives" / split)\n\n    def augmenter(self, seed: int, cfg: AugmentConfig | None = None, window: int | None = None) -> Augmenter:\n        noise = NoiseBank.load(self.shared / "noise" / "bank.npy")\n        rirs = load_rirs(self.asset("rir.zip"))\n        return Augmenter(cfg or self.cfg.augment, noise, rirs, window or self.cfg.window_samples, seed)\n\n    def clips(self, phrase_id: str, kind: str) -> ClipSet:\n        """The phrase\'s clips of one kind from tts.voices and every enabled voice group that has generated them."""\n        sets = [(name, ClipSet.load(path)) for name, path in self.clip_paths(phrase_id, kind) if path.exists()]\n        if len(sets) == 1:\n            return sets[0][1]\n        parts = [c for _, c in sets]\n        return ClipSet.from_clips([c.clip(i) for c in parts for i in range(len(c))],\n                                  np.concatenate([c.speakers for c in parts]),\n                                  np.concatenate([c.texts for c in parts]), parts[0].text_list)\n\n    def clip_paths(self, phrase_id: str, kind: str) -> list[tuple[str, Path]]:\n        main = ("main", self.work / "clips" / f"{phrase_id}_{kind}.npz")\n        return [main] + [(grp.name, self.work / "clips" / f"{phrase_id}_{kind}@{grp.name}.npz")\n                         for _, grp in self.cfg.tts.active_groups]\n\n    def clips_signature(self, phrase_id: str) -> str:\n        """Which clip sets a phrase\'s features were made from, so adding a voice group recomputes them."""\n        sets = []\n        for kind in ("pos", "adv"):\n            for name, path in self.clip_paths(phrase_id, kind):\n                if path.exists():\n                    with np.load(path) as z:\n                        sets.append(f"{kind}/{name}:{len(z[\'offsets\']) - 1}")\n        return ",".join(sets)\n\n    def hours(self) -> dict[str, float]:\n        out = {}\n        for split in ("train", "val"):\n            store = self.store(split)\n            for src in self.cfg.negatives:\n                if src.split == split:\n                    out[src.name] = store.hours(src.name)\n        return out\n\n    def done(self, step: str) -> bool:\n        return (self.work / "steps" / f"{step}.done").exists()\n\n    def mark(self, step: str, summary: dict) -> None:\n        (self.work / "steps" / f"{step}.done").write_text(json.dumps(summary, indent=2, default=str))\n\n    # ---- steps\n\n    def setup(self) -> dict:\n        cfg = self.cfg\n        download(cfg.melspectrogram_url, self.asset("melspectrogram.onnx"))\n        download(cfg.embedding_url, self.asset("embedding_model.onnx"))\n        download(cfg.rir_url, self.asset("rir.zip"))\n        download(cfg.cmudict_url, self.asset("cmudict.dict"))\n        piper_voices = cfg.tts.voices + [v for _, grp in cfg.tts.active_groups if grp.engine == "piper" for v in grp.voices]\n        for v, (model, config) in zip(piper_voices, self.voice_paths(piper_voices)):\n            download(v.model_url, model)\n            download(v.config_url, config)\n        rirs = load_rirs(self.asset("rir.zip"))\n        fx = self.features()\n        probe = fx.embed(np.zeros(cfg.window_samples, dtype=np.int16))\n        if probe.shape != (16, 96):\n            raise RuntimeError(f"feature models produced {probe.shape} for a {cfg.window_samples}-sample window; expected (16, 96)")\n        return {"rirs": len(rirs), "voices": [v.name for v in cfg.tts.voices]}\n\n    def preview(self, per_text: int = 2) -> dict:\n        """A few samples per TTS spelling, to check pronunciation before generating thousands."""\n        rng = np.random.default_rng(0)\n        files = []\n        for synth, (path, _) in zip(self.synths(), self.voice_paths()):\n            for p in self.cfg.phrases:\n                for t, text in enumerate(p.tts_texts):\n                    for k in range(per_text):\n                        audio = synth.synth(text, int(rng.integers(synth.num_speakers)), 1.0, 0.667, 0.8)\n                        out = self.work / "preview" / f"{p.id}_{t}_{k}_{path.stem}.wav"\n                        write_wav(out, audio)\n                        files.append(str(out))\n        return {"files": files}\n\n    def tts(self) -> dict:\n        """Synthesises every missing part, in parallel when tts.workers > 1, then merges parts per phrase and kind."""\n        cfg = self.cfg\n        size = cfg.tts.part_size\n        jobs, groups = [], []\n        for p in cfg.phrases:\n            for kind, texts, n in (("pos", p.tts_texts, cfg.tts.positives_per_phrase),\n                                   ("adv", p.adversarial_texts, cfg.tts.adversarial_per_phrase)):\n                final = self.work / "clips" / f"{p.id}_{kind}.npz"\n                parts_dir = self.work / "clips" / f"{p.id}_{kind}_parts"\n                groups.append((p.id, kind, texts, final, parts_dir))\n                if final.exists():\n                    continue\n                parts_dir.mkdir(exist_ok=True)\n                n_parts = -(-n // size)\n                for part in range(n_parts):\n                    part_path = parts_dir / f"part_{part:03d}.npz"\n                    if not part_path.exists():\n                        seed = [cfg.train.seed, hash_id(p.id), 0 if kind == "pos" else 1, part]\n                        label = f"{p.id} {kind}: part {part + 1}/{n_parts}"\n                        jobs.append((texts, min(size, n - part * size), cfg.tts, seed, str(part_path), label))\n\n        voices = [(str(m), str(c)) for m, c in self.voice_paths()]\n        started = time.time()\n        if cfg.tts.workers > 1 and len(jobs) > 1:\n            ctx = multiprocessing.get_context("spawn")  # ONNX Runtime sessions must not be forked\n            workers = min(cfg.tts.workers, len(jobs))\n            threads = max(1, (os.cpu_count() or workers) // workers)\n            with ProcessPoolExecutor(workers, mp_context=ctx,\n                                     initializer=_init_tts_worker, initargs=(voices, cfg.tts.use_cuda, threads)) as pool:\n                for future in as_completed([pool.submit(_tts_part, job) for job in jobs]):\n                    label, count, seconds = future.result()\n                    print(f"{label} ({count} clips, {seconds:.0f}s; {time.time() - started:.0f}s elapsed)", flush=True)\n        elif jobs:\n            _init_tts_worker(voices, cfg.tts.use_cuda)\n            for job in jobs:\n                label, count, seconds = _tts_part(job)\n                print(f"{label} ({count} clips, {seconds:.0f}s)", flush=True)\n\n        self.tts_groups()\n        summary = {}\n        for phrase_id, kind, texts, final, parts_dir in groups:\n            if not final.exists():\n                parts = [ClipSet.load(f) for f in sorted(parts_dir.glob("part_*.npz"))]\n                ClipSet.from_clips(\n                    [c.clip(i) for c in parts for i in range(len(c))],\n                    np.concatenate([c.speakers for c in parts]),\n                    np.concatenate([c.texts for c in parts]),\n                    texts,\n                ).save(final)\n            summary[f"{phrase_id}_{kind}"] = len(ClipSet.load(final))\n        return summary\n\n    def tts_groups(self) -> None:\n        """Clips for each enabled voice group that does not have them yet, saved per group beside the main clips."""\n        cfg = self.cfg\n        size = cfg.tts.part_size\n        for g, grp in cfg.tts.active_groups:\n            todo = []\n            for p in cfg.phrases:\n                for kind, texts in (("pos", p.tts_texts), ("adv", p.adversarial_texts)):\n                    final = self.work / "clips" / f"{p.id}_{kind}@{grp.name}.npz"\n                    if not final.exists():\n                        todo.append((p, kind, texts, final))\n            if not todo:\n                continue\n            if grp.engine == "parler":\n                synth = ParlerSynth(grp.voices[0].model_url, parler_descriptions(grp.speakers, seed=g), cfg.tts.use_cuda)\n            elif grp.engine == "piper":\n                # A run set up before this group was added has not downloaded its voices yet.\n                for v, (model, config) in zip(grp.voices, self.voice_paths(grp.voices)):\n                    download(v.model_url, model)\n                    download(v.config_url, config)\n                _init_tts_worker([(str(m), str(c)) for m, c in self.voice_paths(grp.voices)], cfg.tts.use_cuda)\n            else:\n                raise ValueError(f"voice group {grp.name}: unknown engine {grp.engine!r}; use \'piper\' or \'parler\'")\n            for p, kind, texts, final in todo:\n                parts_dir = final.with_name(final.stem + "_parts")\n                parts_dir.mkdir(exist_ok=True)\n                n = grp.clips_per_phrase\n                n_parts = -(-n // size)\n                for part in range(n_parts):\n                    part_path = parts_dir / f"part_{part:03d}.npz"\n                    if part_path.exists():\n                        continue\n                    started = time.time()\n                    rng = np.random.default_rng([cfg.train.seed, hash_id(p.id), 0 if kind == "pos" else 1, part, g])\n                    count = min(size, n - part * size)\n                    if grp.engine == "parler":\n                        def report(done, total, label=f"{grp.name} {p.id} {kind}: part {part + 1}/{n_parts}"):\n                            if done % 240 < 24 or done == total:  # about every 10 batches, so it never looks stuck\n                                print(f"  {label}: {done}/{total} clips ({time.time() - started:.0f}s)", flush=True)\n\n                        clips = generate_parler(synth, texts, count, rng, speaker_offset=g * GROUP_SPEAKERS, progress=report)\n                    else:\n                        clips = generate(_TTS_SYNTHS, texts, count, cfg.tts, rng, speaker_offset=g * GROUP_SPEAKERS)\n                    clips.save(part_path)\n                    print(f"{grp.name} {p.id} {kind}: part {part + 1}/{n_parts} ({len(clips)} clips, "\n                          f"{time.time() - started:.0f}s)", flush=True)\n                parts = [ClipSet.load(f) for f in sorted(parts_dir.glob("part_*.npz"))]\n                ClipSet.from_clips([c.clip(i) for c in parts for i in range(len(c))],\n                                   np.concatenate([c.speakers for c in parts]),\n                                   np.concatenate([c.texts for c in parts]), texts).save(final)\n\n    def negatives(self) -> dict:\n        cfg = self.cfg\n        fx = self.features()\n        noise_dir = self.shared / "noise" / "parts"\n        noise_dir.mkdir(parents=True, exist_ok=True)\n        noise_samples = sum(np.load(f, mmap_mode="r").shape[0] for f in noise_dir.glob("*.npy"))\n        noise_target = int(cfg.augment.noise_bank_hours * 3600 * SAMPLE_RATE)\n        noise_buf: list[np.ndarray] = []\n\n        def flush_noise():\n            nonlocal noise_buf\n            if noise_buf:\n                np.save(noise_dir / f"part_{len(list(noise_dir.glob(\'*.npy\'))):04d}.npy", np.concatenate(noise_buf))\n                noise_buf = []\n\n        for src in cfg.negatives:\n            if self.negatives_split and src.split != self.negatives_split:\n                continue\n            store = self.store(src.split)\n            for key, items in self._negative_streams(src, store):\n                last_report = time.time()\n                for index, name, audio in items:\n                    # Whole recordings can be hours long (VoxPopuli sessions): keep only what the budget needs.\n                    have = store.hours(src.name) + store._buf_frames / FRAMES_PER_HOUR\n                    if src.kind == "embeddings":  # features computed elsewhere; the audio never came here\n                        emb = audio[: int((src.max_hours - have) * FRAMES_PER_HOUR)].astype(np.float32)\n                        audio = np.empty(0, np.int16)\n                    else:\n                        audio = audio[: int((src.max_hours - have) * 3600 * SAMPLE_RATE) + MIN_EMBED_SAMPLES]\n                        emb = fx.embed_long(audio)\n                    if emb.shape[0]:\n                        store.append(src.name, key, index, emb)\n                    if audio.shape[0] and noise_samples < noise_target and any(s in name for s in src.noise_bank):\n                        noise_buf.append(audio)\n                        noise_samples += audio.shape[0]\n                        if sum(a.shape[0] for a in noise_buf) > 5 * 60 * SAMPLE_RATE:\n                            flush_noise()\n                    if time.time() - last_report > 60:\n                        print(f"  {src.name}: {store.hours(src.name) + store._buf_frames / FRAMES_PER_HOUR:.2f} h")\n                        last_report = time.time()\n                    if store.hours(src.name) + store._buf_frames / FRAMES_PER_HOUR >= src.max_hours:\n                        break\n                else:\n                    store.mark_exhausted(src.name, key)\n                store.flush()\n                flush_noise()\n                if src.download_first:\n                    self._archive_copy(src, key).unlink(missing_ok=True)\n\n        parts = [np.load(f) for f in sorted(noise_dir.glob("*.npy"))]\n        rng = np.random.default_rng(cfg.train.seed)\n        bank = np.concatenate(parts + [colored_noise(120, rng)])\n        NoiseBank(bank).save(self.shared / "noise" / "bank.npy")\n        hours = self.hours()\n        for split in ("train", "val"):\n            if self.store(split).hours() == 0:\n                raise RuntimeError(f"no {split} negative audio was collected; check the negatives sources")\n        return {"hours": hours, "noise_bank_hours": bank.shape[0] / SAMPLE_RATE / 3600}\n\n    def _negative_streams(self, src, store: EmbeddingStore):\n        """(progress key, audio iterator) for each part of a source still to read."""\n        if src.kind == "files":\n            parts = [("files", lambda skip: iter_file_audio(src.urls, skip_members=skip))]\n        elif src.kind == "embeddings":\n            folder = Path(src.urls[0])\n            files = sorted(folder.glob("*.npy")) if folder.is_dir() else []\n            if not files:\n                print(f"{src.name}: no feature files in {folder}; skipping")\n                return\n            parts = [(str(folder), lambda skip: ((i, f.name, np.load(f)) for i, f in enumerate(files) if i >= skip))]\n        elif src.kind == "tar":\n            accept = (lambda name: not src.include or any(s in name for s in src.include))\n            parts = [(url, lambda skip, url=url: iter_tar_audio(url, accept, skip_members=skip, local=self._fetch(src, url)))\n                     for url in src.urls]\n        else:\n            raise ValueError(f"{src.name}: unknown negatives kind {src.kind!r}; use \'tar\', \'files\' or \'embeddings\'")\n        for key, open_part in parts:\n            if store.hours(src.name) >= src.max_hours:\n                return\n            if store.exhausted(src.name, key):\n                continue\n            skip = store.source_progress(src.name)["next_member"].get(key, 0)\n            where = {"files": f"{len(src.urls)} files from file {skip}", "embeddings": f"features in {key} from file {skip}"}.get(\n                src.kind, f"{key} from member {skip}")\n            print(f"{src.name}: streaming {where} ({store.hours(src.name):.1f}/{src.max_hours} h)")\n            yield key, open_part(skip)\n\n    def _archive_copy(self, src, url: str) -> Path:\n        # Local disk, not the work folder: in Colab that is Google Drive, and GBs written there can drop the mount.\n        return Path(tempfile.gettempdir()) / "wakeword_downloads" / src.name / url.rstrip("/").rsplit("/", 1)[-1]\n\n    def _fetch(self, src, url: str) -> Path | None:\n        if not src.download_first:\n            return None\n        path = self._archive_copy(src, url)\n        print(f"{src.name}: downloading {url} to disk first (resumes if interrupted)")\n        download(url, path)\n        return path\n\n    def features_step(self) -> dict:\n        cfg = self.cfg\n        fx = self.features()\n        summary = {}\n        for p in cfg.phrases:\n            aug = self.augmenter(seed=hash_id(p.id))\n            signature = self.clips_signature(p.id)\n            sig_path = self.work / "features" / f"{p.id}.clips.txt"\n            # Features made before signatures existed came from the main clips alone.\n            old = sig_path.read_text() if sig_path.exists() else self.clips_signature_main(p.id)\n            if old != signature:\n                for kind in ("pos", "adv", "valpos"):\n                    (self.work / "features" / f"{p.id}_{kind}.npy").unlink(missing_ok=True)\n            pos = self.clips(p.id, "pos")\n            held = held_out_mask(pos.speakers, cfg.tts)\n            jobs = {\n                "pos": (pos.subset(~held), cfg.augment.copies_per_clip),\n                "adv": (self.clips(p.id, "adv"), cfg.augment.copies_per_clip),\n                "valpos": (pos.subset(held), 1),\n            }\n            for kind, (clips, copies) in jobs.items():\n                out = self.work / "features" / f"{p.id}_{kind}.npy"\n                if out.exists():\n                    summary[f"{p.id}_{kind}"] = int(np.load(out, mmap_mode="r").shape[0])\n                    continue\n                feats = []\n                for start in range(0, len(clips), 512):\n                    windows = np.stack([aug(clips.clip(i)) for _ in range(copies)\n                                        for i in range(start, min(start + 512, len(clips)))])\n                    feats.append(fx.embed_batch(windows).astype(np.float16))\n                    print(f"{p.id} {kind}: {min(start + 512, len(clips))}/{len(clips)} clips")\n                data = np.concatenate(feats) if feats else np.empty((0, 16, 96), np.float16)\n                np.save(out, data)\n                summary[f"{p.id}_{kind}"] = int(data.shape[0])\n            sig_path.write_text(signature)\n        return summary\n\n    def clips_signature_main(self, phrase_id: str) -> str:\n        sets = []\n        for kind in ("pos", "adv"):\n            path = self.work / "clips" / f"{phrase_id}_{kind}.npz"\n            if path.exists():\n                with np.load(path) as z:\n                    sets.append(f"{kind}/main:{len(z[\'offsets\']) - 1}")\n        return ",".join(sets)\n\n    def train_step(self) -> dict:\n        import torch\n\n        from .trainer import train\n\n        cfg = self.cfg\n        cap = int(cfg.train.max_negative_hours * FRAMES_PER_HOUR) or None\n        neg = self.store("train").load(cap)\n        val_neg = self.store("val").load()\n        summary = {}\n        for p in cfg.phrases:\n            out = self.work / "models" / f"{p.id}.pt"\n            if out.exists():\n                continue\n            print(f"training {p.display}: negatives {neg.shape[0] / FRAMES_PER_HOUR:.1f} h, validation {val_neg.shape[0] / FRAMES_PER_HOUR:.1f} h")\n            model, history = train(\n                np.load(self.work / "features" / f"{p.id}_pos.npy"),\n                np.load(self.work / "features" / f"{p.id}_adv.npy"),\n                neg,\n                np.load(self.work / "features" / f"{p.id}_valpos.npy"),\n                val_neg,\n                cfg.train,\n                cfg.eval,\n            )\n            torch.save({"state": model.state_dict(), "layer_size": cfg.train.layer_size, "n_blocks": cfg.train.n_blocks}, out)\n            (self.work / "models" / f"{p.id}_history.json").write_text(json.dumps(history, indent=1))\n            summary[p.id] = history[-1] if history else {}\n        return summary\n\n    def load_model(self, phrase_id: str):\n        import torch\n\n        from .model import WakeWordDNN\n\n        ckpt = torch.load(self.work / "models" / f"{phrase_id}.pt", map_location="cpu")\n        model = WakeWordDNN(ckpt["layer_size"], ckpt["n_blocks"])\n        model.load_state_dict(ckpt["state"])\n        return model.eval()\n\n    def eval_audio(self, phrase_id: str) -> list[np.ndarray]:\n        """Held-out-speaker positives, each in a 4 s noisy window ending 1 s after the phrase."""\n        cfg = self.cfg\n        eval_aug = AugmentConfig(\n            copies_per_clip=1, snr_db=cfg.eval.val_positive_snr_db, noise_prob=1.0, rir_prob=0.3,\n            gain_db=(-6.0, 6.0), band_limit_prob=0.2, speed_prob=0.0, trailing_seconds=(1.0, 1.0),\n        )\n        aug = self.augmenter(seed=hash_id(phrase_id) + 7, cfg=eval_aug, window=EVAL_WINDOW)\n        pos = self.clips(phrase_id, "pos")\n        held = pos.subset(held_out_mask(pos.speakers, cfg.tts))\n        return [aug(held.clip(i), end_offset=EVAL_TRAILING) for i in range(len(held))]\n\n    def tts_seconds(self) -> dict[tuple[int, str, str], float]:\n        """(voice ref, manifest kind, split) -> seconds of generated clips, for DATASET_MANIFEST.csv.\n\n        A voice ref is speaker id // 100000: group * 10 + voice index (TtsConfig.voice resolves it).\n        """\n        out: dict[tuple[int, str, str], float] = {}\n        for p in self.cfg.phrases:\n            for kind, manifest_kind in (("pos", "positive"), ("adv", "hard_negative")):\n                for _, path in self.clip_paths(p.id, kind):\n                    if path.exists():\n                        self._add_tts_seconds(out, path, kind, manifest_kind)\n        return out\n\n    def _add_tts_seconds(self, out: dict, path: Path, kind: str, manifest_kind: str) -> None:\n        with np.load(path) as z:  # reads only these two arrays, not the audio\n            lengths, speakers = np.diff(z["offsets"]), z["speakers"]\n        # Held-out speakers\' positives are the dev set; every near-miss clip is trained on.\n        held = held_out_mask(speakers, self.cfg.tts) if kind == "pos" else np.zeros(len(speakers), bool)\n        for voice in np.unique(speakers // 100000):\n            for split, mask in (("train", ~held), ("dev", held)):\n                sel = (speakers // 100000 == voice) & mask\n                key = (int(voice), manifest_kind, split)\n                out[key] = out.get(key, 0.0) + float(lengths[sel].sum()) / SAMPLE_RATE\n\n    def rir_seconds(self) -> float:\n        return sum(len(r) for r in load_rirs(self.asset("rir.zip"))) / SAMPLE_RATE\n\n    def evaluate_step(self) -> dict:\n        from .evaluate import evaluate_phrase, stream_scores\n\n        cfg = self.cfg\n        fx = self.features()\n        val_neg = self.store("val").load()\n        summary = {}\n        for p in cfg.phrases:\n            model = self.load_model(p.id)\n            audio = self.eval_audio(p.id)\n            if not audio:\n                raise RuntimeError(f"{p.id}: no held-out positives; generate more clips or lower holdout_every_nth_speaker")\n            feats = fx.embed_batch(np.stack(audio))\n            positive_scores = [stream_scores(model, f, "cpu") for f in feats]\n            negative_scores = stream_scores(model, val_neg, "cpu")\n            ev = evaluate_phrase(p.id, positive_scores, negative_scores, cfg.eval)\n            (self.work / "eval" / f"{p.id}.json").write_text(json.dumps(ev.to_json(), indent=1))\n            summary[p.id] = {"threshold": ev.default_threshold, "recall": ev.default_recall,\n                             "fa_per_hour": ev.default_fa_per_hour, "negative_hours": ev.negative_hours}\n            print(f"{p.display}: sensitivity {ev.default_sensitivity} -> threshold {ev.default_threshold:.3f}, "\n                  f"recall {ev.default_recall:.1%}, {ev.default_fa_per_hour:.2f} false accepts/hour "\n                  f"over {ev.negative_hours:.1f} h")\n        return summary\n\n    def test_step(self) -> dict:\n        """False accepts on your own recordings (cfg.test.folder) at every calibrated sensitivity.\n\n        Writes eval/<phrase>_test.json (totals, which go in the package) and eval/test_fires.csv (file and\n        time of every fire at the most sensitive setting, to check by ear; it stays out of the package).\n        """\n        from eval.metrics import poisson_upper_bound\n\n        from .evaluate import count_fires, stream_scores\n        from .testset import clock, decode_media, media_files, score_seconds\n\n        cfg = self.cfg\n        folder = Path(cfg.test.folder) if cfg.test.folder else None\n        files = media_files(folder) if folder and folder.is_dir() else []\n        if not files:\n            print(f"no recordings to test on ({cfg.test.folder or \'test.folder is not set\'}); skipping")\n            return {"skipped": True}\n        fx = self.features()\n        cache = self.work / "test" / "features"  # embeddings only, keyed by a hash: no audio or names kept\n        cache.mkdir(parents=True, exist_ok=True)\n        streams = []\n        for i, f in enumerate(files):\n            rel = f.relative_to(folder).as_posix()\n            c = cache / (hashlib.sha1(rel.encode()).hexdigest()[:16] + ".npy")\n            if c.exists():\n                emb = np.load(c)\n            else:\n                emb = fx.embed_long(decode_media(f)).astype(np.float16)\n                with open(c.with_suffix(".tmp"), "wb") as out:\n                    np.save(out, emb)\n                c.with_suffix(".tmp").replace(c)\n            streams.append((rel, emb))\n            print(f"  {i + 1}/{len(files)} recordings, {sum(e.shape[0] for _, e in streams) / FRAMES_PER_HOUR:.1f} h")\n        hours = sum(e.shape[0] for _, e in streams) / FRAMES_PER_HOUR\n\n        def fire_frames(scores: np.ndarray, threshold: float) -> np.ndarray:\n            return count_fires(scores, threshold, cfg.eval.consecutive_frames, cfg.eval.refractory_seconds)\n\n        summary, fires = {}, []\n        for p in cfg.phrases:\n            model = self.load_model(p.id)\n            table = json.loads((self.work / "eval" / f"{p.id}.json").read_text())["sensitivity_table"]\n            scores = [(rel, stream_scores(model, emb, "cpu")) for rel, emb in streams]\n            rows = []\n            for s, t in table:\n                n = sum(len(fire_frames(sc, t)) for _, sc in scores)\n                rows.append({"sensitivity": s, "threshold": t, "false_accepts": n,\n                             "fa_per_hour": n / hours, "fa_per_hour_upper": poisson_upper_bound(n) / hours})\n            lowest = min(t for _, t in table)\n            for rel, sc in scores:\n                for k in fire_frames(sc, lowest).tolist():\n                    fires.append({"keyword": p.id, "file": rel, "time": clock(score_seconds(k)), "score": round(float(sc[k]), 3)})\n            result = {"files": len(files), "hours": hours, "model_sha256": self.model_hash(p.id), "by_sensitivity": rows}\n            (self.work / "eval" / f"{p.id}_test.json").write_text(json.dumps(result, indent=1))\n            default = next(r for r in rows if r["sensitivity"] == cfg.eval.default_sensitivity)\n            summary[p.id] = {k: default[k] for k in ("threshold", "false_accepts", "fa_per_hour", "fa_per_hour_upper")}\n            print(f"{p.display}: {default[\'false_accepts\']} false accepts in {hours:.1f} h of your recordings at sensitivity "\n                  f"{cfg.eval.default_sensitivity} ({default[\'fa_per_hour\']:.2f}/h, at most {default[\'fa_per_hour_upper\']:.2f}/h)")\n        with open(self.work / "eval" / "test_fires.csv", "w", newline="", encoding="utf-8") as f:\n            w = csv.DictWriter(f, fieldnames=["keyword", "file", "time", "score"])\n            w.writeheader()\n            w.writerows(sorted(fires, key=lambda r: (r["file"], r["time"], r["keyword"])))\n        print(f"{len(fires)} fires at the most sensitive setting listed in {self.work / \'eval\' / \'test_fires.csv\'}: "\n              "listen to each to check nobody actually said the wake phrase")\n        return {"hours": hours, "files": len(files), **summary}\n\n    def model_hash(self, phrase_id: str) -> str:\n        return hashlib.sha256((self.work / "models" / f"{phrase_id}.pt").read_bytes()).hexdigest()\n\n    def current_tests(self, evaluations: dict[str, dict]) -> dict[str, dict]:\n        """Test results that still match the models and calibration; stale ones (after a retrain) are left out."""\n        out = {}\n        for p in self.cfg.phrases:\n            path = self.work / "eval" / f"{p.id}_test.json"\n            if not path.exists():\n                continue\n            test = json.loads(path.read_text())\n            thresholds = [r["threshold"] for r in test["by_sensitivity"]]\n            if test["model_sha256"] == self.model_hash(p.id) and thresholds == [t for _, t in evaluations[p.id]["sensitivity_table"]]:\n                out[p.id] = test\n            else:\n                print(f"{p.id}: test results are from older models or calibration; re-run the test step to include them")\n        return out if len(out) == len(self.cfg.phrases) else {}\n\n    def export_step(self) -> dict:\n        from .export import check_parity, export_onnx, golden_vectors, write_package\n\n        cfg = self.cfg\n        out = self.work / "export" / "wakeword_models"\n        out.mkdir(parents=True, exist_ok=True)\n        classifiers, parity = {}, {}\n        for p in cfg.phrases:\n            model = self.load_model(p.id)\n            path = self.work / "export" / f"{p.id}.onnx"\n            export_onnx(model, path)\n            windows = np.load(self.work / "features" / f"{p.id}_valpos.npy")[:256]\n            parity[p.id] = check_parity(model, path, windows)\n            classifiers[p.id] = path\n        golden = golden_vectors(self.features(), classifiers, self.eval_audio(cfg.phrases[0].id)[0])\n        evaluations = {p.id: json.loads((self.work / "eval" / f"{p.id}.json").read_text()) for p in cfg.phrases}\n        hours = self.hours()\n        def used(split: str) -> str:\n            return ", ".join(f"{src.name} ({hours.get(src.name, 0):.0f} h)" for src in cfg.negatives\n                             if src.split == split and hours.get(src.name, 0) > 0)\n\n        val_sources = used("val")\n        notes = [\n            f"Provisional v0: positives are synthetic (Piper) only; training negatives are {used(\'train\')}.",\n            f"Recall is measured on held-out synthetic voices and FA/hour on {val_sources}, so real-world numbers will differ. "\n            "Re-measure on the real-recording test set (plan 5.3) before release.",\n            "All training data is commercially licensed; see DATASET_MANIFEST.csv.",\n        ]\n        zip_path = write_package(\n            cfg, out,\n            {"melspectrogram": self.asset("melspectrogram.onnx"), "embedding": self.asset("embedding_model.onnx")},\n            classifiers, evaluations, hours, golden, notes, self.tts_seconds(), self.rir_seconds(),\n            self.current_tests(evaluations),\n        )\n        print(f"model package: {zip_path}")\n        return {"zip": str(zip_path), "onnx_parity_max_diff": parity}\n\n\n_TTS_SYNTHS: list[PiperSynth] | None = None\n\n\ndef _init_tts_worker(voices: list[tuple[str, str]], use_cuda: bool, threads: int = 0) -> None:\n    global _TTS_SYNTHS\n    if threads:\n        # Piper builds its own SessionOptions with every core; with several workers that oversubscribes the CPU\n        # badly (8 workers x 16 threads), so each worker gets its share of the cores instead.\n        import onnxruntime\n\n        base = onnxruntime.SessionOptions\n\n        def limited() -> onnxruntime.SessionOptions:\n            options = base()\n            options.intra_op_num_threads = threads\n            options.inter_op_num_threads = 1\n            return options\n\n        onnxruntime.SessionOptions = limited  # type: ignore[assignment]\n    _TTS_SYNTHS = [PiperSynth(Path(m), Path(c), use_cuda) for m, c in voices]\n\n\ndef _tts_part(job: tuple) -> tuple[str, int, float]:\n    texts, count, tts_cfg, seed, part_path, label = job\n    started = time.time()\n    clips = generate(_TTS_SYNTHS, texts, count, tts_cfg, np.random.default_rng(seed))\n    clips.save(Path(part_path))\n    return label, len(clips), time.time() - started\n\n\ndef hash_id(s: str) -> int:\n    return int.from_bytes(s.encode("utf-8")[:8].ljust(8, b"\\0"), "little") % (2**31)\n\n\ndef write_wav(path: Path, audio: np.ndarray) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with wave.open(str(path), "wb") as w:\n        w.setnchannels(1)\n        w.setsampwidth(2)\n        w.setframerate(SAMPLE_RATE)\n        w.writeframes(audio.astype(np.int16).tobytes())\n\n\ndef run(config: Path, work: Path, steps: list[str], force: bool = False, overrides: dict | None = None,\n        negatives_split: str | None = None) -> None:\n    cfg = load_config(config, overrides)\n    r = Run(cfg, work, negatives_split)\n    handlers = {\n        "setup": r.setup, "preview": r.preview, "tts": r.tts, "negatives": r.negatives,\n        "features": r.features_step, "train": r.train_step, "evaluate": r.evaluate_step, "export": r.export_step,\n    }\n    for step in steps:\n        if r.done(step) and not force:\n            print(f"[{step}] already done")\n            continue\n        print(f"[{step}] starting")\n        started = time.time()\n        summary = handlers[step]()\n        r.mark(step, {"seconds": round(time.time() - started), **(summary or {})})\n        print(f"[{step}] done in {time.time() - started:.0f}s")\n\n\ndef main(argv: list[str] | None = None) -> None:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--config", required=True, type=Path)\n    ap.add_argument("--work", required=True, type=Path)\n    ap.add_argument("--steps", default="all", help="comma-separated steps, or \'all\'")\n    ap.add_argument("--force", action="store_true", help="re-run steps already marked done")\n    ap.add_argument("--set", action="append", default=[], metavar="KEY=VALUE", help="override a config value, e.g. train.steps=500")\n    ap.add_argument("--negatives-split", choices=["train", "val"], help="limit the negatives step to one split\'s sources")\n    args = ap.parse_args(argv)\n    steps = STEPS if args.steps == "all" else [s.strip() for s in args.steps.split(",")]\n    unknown = [s for s in steps if s not in STEPS]\n    if unknown:\n        ap.error(f"unknown steps {unknown}; choose from {STEPS}")\n    overrides = {k: yaml.safe_load(v) for k, v in (s.split("=", 1) for s in args.set)}\n    run(args.config, args.work, steps, args.force, overrides, args.negatives_split)\n\n\nif __name__ == "__main__":\n    main()\n',
'wakeword_train/sources.py': '"""Downloads and streaming readers for openly licensed audio."""\n\nfrom __future__ import annotations\n\nimport io\nimport tarfile\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom pathlib import Path\nfrom typing import Callable, Iterator\n\nimport numpy as np\nimport requests\nimport soundfile as sf\nimport urllib3\nfrom scipy.signal import resample_poly\n\nfrom . import SAMPLE_RATE\n\nAUDIO_EXTENSIONS = (".flac", ".wav", ".ogg")\n\n\ndef download(url: str, dest: Path, retries: int = 5) -> Path:\n    """Downloads with resume; skips if already complete."""\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    part = dest.with_suffix(dest.suffix + ".part")\n    for attempt in range(retries):\n        try:\n            head = requests.head(url, allow_redirects=True, timeout=30)\n            total = int(head.headers.get("content-length", 0))\n            if dest.exists() and (total == 0 or dest.stat().st_size == total):\n                return dest\n            done = part.stat().st_size if part.exists() else 0\n            headers = {"Range": f"bytes={done}-"} if done else {}\n            with requests.get(url, headers=headers, stream=True, timeout=60) as r:\n                r.raise_for_status()\n                mode = "ab" if done and r.status_code == 206 else "wb"\n                with open(part, mode) as f:\n                    for block in r.iter_content(1 << 20):\n                        f.write(block)\n            part.replace(dest)\n            return dest\n        except (requests.RequestException, OSError) as e:\n            if attempt == retries - 1:\n                raise\n            print(f"download {url} failed ({e}); retrying")\n            time.sleep(5 * (attempt + 1))\n    return dest\n\n\ndef to_16k_mono_int16(audio: np.ndarray, sr: int) -> np.ndarray:\n    """int16 input keeps its scale; float input is taken as [-1, 1] full scale.\n\n    The scale is decided by the input dtype, never by the values: resampling a clip normalised to a peak\n    of exactly 1.0 can overshoot slightly, and a value-based check then skipped scaling and produced\n    near-silent clips.\n    """\n    if audio.dtype == np.int16 and sr == SAMPLE_RATE and (audio.ndim == 1 or audio.shape[1] == 1):\n        # Already the target format: no float copies, which for a 14-hour VoxPopuli session would need ~10 GB.\n        return audio.reshape(-1)\n    x = audio.astype(np.float32)\n    if np.issubdtype(audio.dtype, np.floating):\n        x = x * 32767.0\n    if x.ndim == 2:\n        x = x.mean(axis=1)\n    if sr != SAMPLE_RATE:\n        g = np.gcd(sr, SAMPLE_RATE)\n        x = resample_poly(x, SAMPLE_RATE // g, sr // g)\n    return np.clip(np.round(x), -32768, 32767).astype(np.int16)\n\n\ndef decode(data: bytes) -> np.ndarray:\n    audio, sr = sf.read(io.BytesIO(data), dtype="int16", always_2d=True)\n    return to_16k_mono_int16(audio, sr)\n\n\n@contextmanager\ndef _open_archive(url: str, local: Path | None):\n    if local is not None:\n        with open(local, "rb") as f:\n            yield f\n    else:\n        with requests.get(url, stream=True, timeout=120) as r:\n            r.raise_for_status()\n            yield r.raw\n\n\ndef iter_tar_audio(\n    url: str,\n    accept: Callable[[str], bool],\n    skip_members: int = 0,\n    retries: int = 5,\n    local: Path | None = None,\n) -> Iterator[tuple[int, str, np.ndarray]]:\n    """Streams a remote .tar/.tar.gz without saving it, yielding (member_index, name, int16 16 kHz audio).\n\n    member_index counts every regular file in archive order, so a crashed run can resume with\n    skip_members=<last index + 1>. Connection drops reconnect and skip ahead automatically.\n    With local, reads that already-downloaded copy of url instead.\n    """\n    index = -1\n    next_needed = skip_members\n    for attempt in range(retries):\n        try:\n            with _open_archive(url, local) as raw:\n                index = -1\n                with tarfile.open(fileobj=raw, mode="r|*") as tf:\n                    for member in tf:\n                        if not member.isfile():\n                            continue\n                        index += 1\n                        if index < next_needed:\n                            continue\n                        name = member.name\n                        if name.lower().endswith(AUDIO_EXTENSIONS) and accept(name):\n                            try:\n                                audio = decode(tf.extractfile(member).read())\n                            except (RuntimeError, sf.LibsndfileError) as e:\n                                print(f"skipping undecodable {name}: {e}")\n                                next_needed = index + 1\n                                continue\n                            next_needed = index + 1\n                            yield index, name, audio\n                        else:\n                            next_needed = index + 1\n            return\n        # Reading r.raw directly raises urllib3\'s own errors (a reset is ProtocolError), not requests\'.\n        except (requests.RequestException, urllib3.exceptions.HTTPError, tarfile.ReadError, OSError, EOFError) as e:\n            if attempt == retries - 1:\n                raise\n            print(f"stream {url} dropped at member {next_needed} ({e}); reconnecting")\n            time.sleep(10 * (attempt + 1))\n\n\ndef iter_file_audio(urls: list[str], skip_members: int = 0, retries: int = 5) -> Iterator[tuple[int, str, np.ndarray]]:\n    """Fetches individual remote audio files, yielding (index in urls, url, int16 16 kHz audio).\n\n    Resumes like iter_tar_audio with skip_members=<last index + 1>. A file that is missing (404) or\n    undecodable is skipped with a message rather than ending the run.\n    """\n    for index in range(skip_members, len(urls)):\n        url = urls[index]\n        for attempt in range(retries):\n            try:\n                r = requests.get(url, timeout=120)\n                if r.status_code == 404:\n                    print(f"skipping missing {url}")\n                    break\n                r.raise_for_status()\n                try:\n                    audio = decode(r.content)\n                except (RuntimeError, sf.LibsndfileError) as e:\n                    print(f"skipping undecodable {url}: {e}")\n                    break\n                yield index, url, audio\n                break\n            except requests.RequestException as e:\n                if attempt == retries - 1:\n                    raise\n                print(f"fetching {url} failed ({e}); retrying")\n                time.sleep(10 * (attempt + 1))\n\n\ndef load_rirs(zip_path: Path, max_seconds: float = 1.0) -> list[np.ndarray]:\n    """Room impulse responses from a zip of wav files, as float32 at 16 kHz, peak-normalised."""\n    rirs = []\n    with zipfile.ZipFile(zip_path) as z:\n        for name in sorted(z.namelist()):\n            if not name.lower().endswith(".wav") or "__MACOSX" in name:\n                continue\n            audio, sr = sf.read(io.BytesIO(z.read(name)), dtype="float32", always_2d=True)\n            x = audio.mean(axis=1)\n            if sr != SAMPLE_RATE:\n                g = np.gcd(sr, SAMPLE_RATE)\n                x = resample_poly(x, SAMPLE_RATE // g, sr // g).astype(np.float32)\n            x = x[np.argmax(np.abs(x)):][: int(max_seconds * SAMPLE_RATE)]  # start at the direct path\n            peak = np.abs(x).max()\n            if peak > 0:\n                rirs.append((x / peak).astype(np.float32))\n    return rirs\n',
'wakeword_train/store.py': '"""Append-only, resumable store of continuous embedding streams (negative audio)."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom . import CLASSIFIER_FRAMES, EMBEDDING_DIM, FRAME_SAMPLES, SAMPLE_RATE\n\nFRAMES_PER_HOUR = 3600 * SAMPLE_RATE / FRAME_SAMPLES  # 45,000\n\n\nclass EmbeddingStore:\n    """Directory of float16 [T, 96] shards plus index.json recording per-source progress.\n\n    Shards are only listed in the index after they are fully written, so a crash loses at most one\n    unflushed shard, and resume restarts each source after its last flushed member.\n    """\n\n    def __init__(self, root: Path, shard_frames: int = 50000):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.shard_frames = shard_frames\n        self.index_path = self.root / "index.json"\n        self.index = json.loads(self.index_path.read_text()) if self.index_path.exists() else {"shards": [], "sources": {}}\n        self._buf: list[np.ndarray] = []\n        self._buf_frames = 0\n        self._pending_progress: dict[str, dict] = {}\n\n    def source_progress(self, source: str) -> dict:\n        return self.index["sources"].get(source, {"next_member": {}, "frames": 0})\n\n    def exhausted(self, source: str, key: str) -> bool:\n        """True once an archive (or a "files" list) was read to the end, so a re-run need not re-stream it."""\n        return key in self.source_progress(source).get("exhausted", [])\n\n    def mark_exhausted(self, source: str, key: str) -> None:\n        progress = self._pending_progress.setdefault(source, json.loads(json.dumps(self.source_progress(source))))\n        progress.setdefault("exhausted", []).append(key)\n        self.flush()\n\n    def hours(self, source: str | None = None) -> float:\n        if source is None:\n            return sum(s["frames"] for s in self.index["shards"]) / FRAMES_PER_HOUR\n        return self.source_progress(source)["frames"] / FRAMES_PER_HOUR\n\n    def append(self, source: str, url: str, member_index: int, embeddings: np.ndarray) -> None:\n        self._buf.append(embeddings.astype(np.float16))\n        self._buf_frames += embeddings.shape[0]\n        progress = self._pending_progress.setdefault(source, json.loads(json.dumps(self.source_progress(source))))\n        progress["next_member"][url] = member_index + 1\n        progress["frames"] += embeddings.shape[0]\n        if self._buf_frames >= self.shard_frames:\n            self.flush()\n\n    def flush(self) -> None:\n        if not self._buf:\n            self._persist_progress()\n            return\n        data = np.concatenate(self._buf)\n        name = f"shard_{len(self.index[\'shards\']):05d}.npy"\n        tmp = self.root / (name + ".tmp")\n        with open(tmp, "wb") as f:\n            np.save(f, data)\n        tmp.replace(self.root / name)\n        self.index["shards"].append({"file": name, "frames": int(data.shape[0])})\n        self._buf, self._buf_frames = [], 0\n        self._persist_progress()\n\n    def _persist_progress(self) -> None:\n        self.index["sources"].update(self._pending_progress)\n        self._pending_progress = {}\n        tmp = self.index_path.with_suffix(".tmp")\n        tmp.write_text(json.dumps(self.index, indent=1))\n        tmp.replace(self.index_path)\n\n    def load(self, max_frames: int | None = None) -> np.ndarray:\n        """All shards concatenated into one float16 [T, 96] array.\n\n        Filled shard by shard into one preallocated array, so peak RAM is the result, not twice it.\n        """\n        shards, total = [], 0\n        for s in self.index["shards"]:\n            if max_frames and total >= max_frames:\n                break\n            shards.append(s)\n            total += s["frames"]\n        n = min(total, max_frames) if max_frames else total\n        out = np.empty((n, EMBEDDING_DIM), dtype=np.float16)\n        pos = 0\n        for s in shards:\n            take = min(s["frames"], n - pos)\n            out[pos:pos + take] = np.load(self.root / s["file"], mmap_mode="r")[:take]\n            pos += take\n        return out\n\n\ndef sample_windows(stream: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:\n    """n random [16, 96] windows from a [T, 96] stream."""\n    starts = rng.integers(0, stream.shape[0] - CLASSIFIER_FRAMES + 1, size=n)\n    return stream[starts[:, None] + np.arange(CLASSIFIER_FRAMES)]\n',
'wakeword_train/test_package.py': '"""Tests a finished model package on your own recordings, on this machine: the audio never leaves it.\n\n    python -m wakeword_train.test_package --package ../models/wakeword --folder "D:/meetings" --out D:/wakeword_test\n    python -m wakeword_train.test_package --package <newer package> --out D:/wakeword_test   # recordings no longer needed\n    python -m wakeword_train.test_package --package ... --out D:/wakeword_test --train-share 0.67 --export-train D:/meetings_train\n\nReports false accepts per hour at every sensitivity in the package\'s calibration, with the detector rules the\napps use (no adaptive offset), and lists every fire at the most sensitive setting in fires.csv to check by ear.\nEmbeddings are cached in --out (with cache/index.json naming each recording), so a newer package is tested on the\nsame recordings in minutes, and without --folder once they are processed: the audio itself can then be deleted.\nRecordings must contain no wake phrase; a meeting where someone says it should be left out.\n\n--train-share splits the recordings once, by whole recording, into a share to train on and the rest to test on, and\nsaves the split in --out/split.json. From then on only the test share is tested, and the split never changes, so the\ntest stays honest. --export-train writes the training share\'s features (not audio) for the "embeddings" negatives source.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport onnxruntime as ort\n\nfrom eval.metrics import poisson_upper_bound\n\nfrom . import CLASSIFIER_FRAMES\nfrom .evaluate import count_fires\nfrom .features import FeatureExtractor\nfrom .store import FRAMES_PER_HOUR\nfrom .testset import clock, decode_media, media_files, score_seconds\n\n\ndef onnx_scores(session: ort.InferenceSession, input_name: str, emb: np.ndarray, batch: int = 8192) -> np.ndarray:\n    """Score for every position of a [T, 96] stream, as the apps compute it (embeddings k..k+15)."""\n    n = emb.shape[0] - CLASSIFIER_FRAMES + 1\n    if n <= 0:\n        return np.empty(0, np.float32)\n    offsets = np.arange(CLASSIFIER_FRAMES)\n    out = []\n    for i in range(0, n, batch):\n        starts = np.arange(i, min(i + batch, n))\n        out.append(session.run(None, {input_name: emb[starts[:, None] + offsets].astype(np.float32)})[0][:, 0])\n    return np.concatenate(out)\n\n\ndef file_hash(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef load_index(cache: Path) -> dict[str, dict]:\n    """cache/index.json: embedding file key -> {"file": recording\'s path in the folder, "features": feature models id}."""\n    path = cache / "index.json"\n    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}\n\n\ndef save_index(cache: Path, index: dict[str, dict]) -> None:\n    tmp = cache / "index.json.tmp"\n    tmp.write_text(json.dumps(index, indent=1, sort_keys=True), encoding="utf-8")\n    tmp.replace(cache / "index.json")\n\n\ndef embeddings(files: list[Path], folder: Path, fx: FeatureExtractor, features_id: str, cache: Path) -> list[tuple[str, np.ndarray]]:\n    cache.mkdir(parents=True, exist_ok=True)\n    index = load_index(cache)\n    streams, started = [], time.time()\n    for i, f in enumerate(files):\n        st = f.stat()\n        rel = f.relative_to(folder).as_posix()\n        key = hashlib.sha1(f"{f.resolve()}|{st.st_size}|{st.st_mtime_ns}|{features_id}".encode()).hexdigest()[:20]\n        c = cache / f"{key}.npy"\n        if c.exists():\n            emb = np.load(c)\n        else:\n            emb = fx.embed_long(decode_media(f)).astype(np.float16)\n            with open(c.with_suffix(".tmp"), "wb") as out:\n                np.save(out, emb)\n            c.with_suffix(".tmp").replace(c)\n        for old in [k for k, v in index.items() if v["file"] == rel and v["features"] == features_id and k != key]:\n            (cache / f"{old}.npy").unlink(missing_ok=True)  # the recording changed since it was processed\n            del index[old]\n        index[key] = {"file": rel, "features": features_id}\n        save_index(cache, index)\n        streams.append((rel, emb))\n        hours = sum(e.shape[0] for _, e in streams) / FRAMES_PER_HOUR\n        print(f"  {i + 1}/{len(files)} recordings, {hours:.1f} h ({(time.time() - started) / 60:.0f} min)", flush=True)\n    return streams\n\n\ndef cached_embeddings(cache: Path, features_id: str) -> list[tuple[str, np.ndarray]]:\n    """Recordings processed earlier with these feature models, without needing the recordings themselves."""\n    index = load_index(cache)\n    entries = sorted((v["file"], k) for k, v in index.items() if v["features"] == features_id and (cache / f"{k}.npy").exists())\n    return [(rel, np.load(cache / f"{k}.npy")) for rel, k in entries]\n\n\ndef make_split(streams: list[tuple[str, np.ndarray]], train_share: float, seed: int = 0) -> dict[str, list[str]]:\n    """Whole recordings, in a fixed random order, go to training until it holds train_share of the hours."""\n    total = sum(e.shape[0] for _, e in streams)\n    train, frames = [], 0\n    for i in np.random.default_rng(seed).permutation(len(streams)).tolist():\n        if frames >= train_share * total:\n            break\n        train.append(streams[i][0])\n        frames += streams[i][1].shape[0]\n    return {"train": sorted(train), "test": sorted(rel for rel, _ in streams if rel not in set(train))}\n\n\ndef export_train(streams: list[tuple[str, np.ndarray]], names: set[str], folder: Path) -> float:\n    """The training share\'s features as .npy files named by a hash, so no recording names leave this machine."""\n    folder.mkdir(parents=True, exist_ok=True)\n    frames = 0\n    for rel, emb in streams:\n        if rel in names:\n            np.save(folder / (hashlib.sha1(rel.encode()).hexdigest()[:16] + ".npy"), emb)\n            frames += emb.shape[0]\n    return frames / FRAMES_PER_HOUR\n\n\ndef main(argv: list[str] | None = None) -> None:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--package", required=True, type=Path, help="unpacked model package (models.json and .onnx files)")\n    ap.add_argument("--folder", type=Path,\n                    help="recordings with no wake phrase in them, searched recursively; omit to reuse those already processed into --out")\n    ap.add_argument("--out", required=True, type=Path, help="where the report, fires.csv and the embedding cache go")\n    ap.add_argument("--train-share", type=float, help="once: put this share of the hours (by whole recording) aside for training")\n    ap.add_argument("--export-train", type=Path, help="write the training share\'s features here, to upload for training")\n    args = ap.parse_args(argv)\n\n    pkg = json.loads((args.package / "models.json").read_text(encoding="utf-8"))\n    feats = pkg["features"]\n    mel, emb_model = args.package / feats["melspectrogram"], args.package / feats["embedding"]\n    features_id = file_hash(mel)[:12] + file_hash(emb_model)[:12]\n    if args.folder:\n        files = media_files(args.folder)\n        if not files:\n            raise SystemExit(f"no audio or video files in {args.folder}")\n        print(f"embedding {len(files)} recordings from {args.folder}")\n        streams = embeddings(files, args.folder, FeatureExtractor(str(mel), str(emb_model)), features_id, args.out / "cache")\n    else:\n        streams = cached_embeddings(args.out / "cache", features_id)\n        if not streams:\n            raise SystemExit(f"no recordings processed with this package\'s feature models in {args.out / \'cache\'}; pass --folder")\n        print(f"using {len(streams)} recordings already processed in {args.out / \'cache\'}")\n    split_path = args.out / "split.json"\n    if args.train_share is not None:\n        if split_path.exists():\n            raise SystemExit(f"{split_path} exists: the split is frozen so the test stays honest. Drop --train-share.")\n        split_path.write_text(json.dumps(make_split(streams, args.train_share), indent=1), encoding="utf-8")\n    split = json.loads(split_path.read_text(encoding="utf-8")) if split_path.exists() else None\n    held_back = ""\n    if split:\n        train_names = set(split["train"])\n        if args.export_train:\n            h = export_train(streams, train_names, args.export_train)\n            print(f"exported features of {len(train_names)} recordings ({h:.1f} h) for training to {args.export_train}")\n        trained = [s_ for s_ in streams if s_[0] in train_names]\n        streams = [s_ for s_ in streams if s_[0] not in train_names]  # never test on what may be trained on\n        held_back = (f" {len(trained)} other recordings ({sum(e.shape[0] for _, e in trained) / FRAMES_PER_HOUR:.1f} h) are "\n                     "set aside for training and not tested on.")\n        if not streams:\n            raise SystemExit("every recording is in the training share; nothing left to test on")\n    elif args.export_train:\n        raise SystemExit("--export-train needs a split: pass --train-share the first time")\n    hours = sum(e.shape[0] for _, e in streams) / FRAMES_PER_HOUR\n\n    results, fires = {}, []\n    for kw in pkg["keywords"]:\n        session = ort.InferenceSession(str(args.package / kw["model"]), providers=["CPUExecutionProvider"])\n        det = kw["detector"]\n        scores = [(rel, onnx_scores(session, kw["input"], e)) for rel, e in streams]\n\n        def fire_frames(sc: np.ndarray, threshold: float) -> np.ndarray:\n            return count_fires(sc, threshold, det["consecutive_frames"], det["refractory_seconds"])\n\n        rows = []\n        for s, t in kw["sensitivity_table"]:\n            n = sum(len(fire_frames(sc, t)) for _, sc in scores)\n            rows.append({"sensitivity": s, "threshold": t, "false_accepts": n,\n                         "fa_per_hour": n / hours, "fa_per_hour_upper": poisson_upper_bound(n) / hours})\n        lowest = min(t for _, t in kw["sensitivity_table"])\n        for rel, sc in scores:\n            for k in fire_frames(sc, lowest).tolist():\n                fires.append({"keyword": kw["id"], "file": rel, "time": clock(score_seconds(k)), "score": round(float(sc[k]), 3)})\n        results[kw["id"]] = {"phrase": kw["phrase"], "default_sensitivity": kw["default_sensitivity"], "by_sensitivity": rows}\n\n    args.out.mkdir(parents=True, exist_ok=True)\n    with open(args.out / "fires.csv", "w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=["keyword", "file", "time", "score"])\n        w.writeheader()\n        w.writerows(sorted(fires, key=lambda r: (r["file"], r["time"], r["keyword"])))\n    summary = {"package_run": pkg.get("run"), "package_created": pkg.get("created"), "recordings": len(streams),\n               "hours": hours, "keywords": results}\n    (args.out / "results.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")\n\n    ids = list(results)\n    lines = [f"# Test on your own recordings: {len(streams)} recordings, {hours:.1f} h", "",\n             f"Package `{pkg.get(\'run\')}` created {pkg.get(\'created\')}. False accepts per hour, 95% upper bound in "\n             f"brackets; plan Section 3 targets at most 0.2 (quiet) to 1.0 (loud).{held_back}", "",\n             "| Sensitivity | " + " | ".join(results[i]["phrase"] for i in ids) + " |", "| --- |" + " --- |" * len(ids)]\n    for j, r in enumerate(results[ids[0]]["by_sensitivity"]):\n        cells = [results[i]["by_sensitivity"][j] for i in ids]\n        lines.append(f"| {r[\'sensitivity\']:.1f} | " + " | ".join(\n            f"{c[\'false_accepts\']} = {c[\'fa_per_hour\']:.2f} ({c[\'fa_per_hour_upper\']:.2f})" for c in cells) + " |")\n    lines += ["", f"{len(fires)} fires at the most sensitive setting are listed in fires.csv: listen to each to check "\n              "whether someone actually said the wake phrase."]\n    report = "\\n".join(lines) + "\\n"\n    (args.out / "REPORT.md").write_text(report, encoding="utf-8")\n    print(report)\n\n\nif __name__ == "__main__":\n    main()\n',
'wakeword_train/testset.py': '"""Your own recordings as the frozen test set (plan 5.3): false accepts on audio the models never saw."""\n\nfrom __future__ import annotations\n\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\n\nfrom . import CLASSIFIER_FRAMES, FRAME_SAMPLES, SAMPLE_RATE\nfrom .features import MIN_EMBED_SAMPLES\nfrom .sources import to_16k_mono_int16\n\nMEDIA_EXTENSIONS = (".mp4", ".m4a", ".webm", ".mkv", ".mov", ".mp3", ".aac", ".wav", ".ogg", ".flac")\n\n\ndef media_files(folder: Path) -> list[Path]:\n    return sorted(p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in MEDIA_EXTENSIONS)\n\n\ndef decode_media(path: Path) -> np.ndarray:\n    """Any audio or video file -> int16 16 kHz mono, through ffmpeg (Meet and Teams save .mp4).\n\n    Without ffmpeg (it ships with Colab), plain audio files still work through soundfile.\n    """\n    if shutil.which("ffmpeg") is None:\n        if Path(path).suffix.lower() in (".wav", ".flac", ".ogg"):\n            audio, sr = sf.read(str(path), dtype="int16", always_2d=True)\n            return to_16k_mono_int16(audio, sr)\n        raise RuntimeError(f"reading {Path(path).suffix} files needs ffmpeg on the PATH: {path}")\n    # Run from the file\'s folder with a bare name: works for a Windows ffmpeg.exe called from WSL too.\n    path = Path(path).resolve()\n    r = subprocess.run(\n        ["ffmpeg", "-nostdin", "-v", "error", "-i", path.name, "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE), "-f", "s16le", "-"],\n        capture_output=True, cwd=path.parent,\n    )\n    if r.returncode != 0:\n        raise RuntimeError(f"ffmpeg could not read {path}: {r.stderr.decode(errors=\'replace\').strip()[-300:]}")\n    return np.frombuffer(r.stdout, dtype=np.int16)\n\n\ndef score_seconds(k: int) -> float:\n    """Where in the recording score k ends: it covers embeddings k..k+15, the last ending at this sample."""\n    return ((k + CLASSIFIER_FRAMES - 1) * FRAME_SAMPLES + MIN_EMBED_SAMPLES) / SAMPLE_RATE\n\n\ndef clock(seconds: float) -> str:\n    s = int(seconds)\n    return f"{s // 3600}:{s // 60 % 60:02d}:{s % 60:02d}"\n',
'wakeword_train/trainer.py': '"""Trains one phrase\'s classifier head on precomputed embeddings."""\n\nfrom __future__ import annotations\n\nimport copy\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom . import CLASSIFIER_FRAMES\nfrom .config import EvalConfig, TrainConfig\nfrom .evaluate import count_fires, stream_scores, window_scores\nfrom .model import WakeWordDNN, average_state_dicts\nfrom .store import FRAMES_PER_HOUR\n\n\ndef _device() -> str:\n    return "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef _quick_metrics(model, val_pos, val_neg, device, eval_cfg: EvalConfig) -> tuple[float, float]:\n    """Recall on held-out positive windows and false accepts/hour (single frame, refractory) at 0.5."""\n    model.eval()\n    recall = float((window_scores(model, val_pos, device) >= 0.5).mean()) if len(val_pos) else 0.0\n    scores = stream_scores(model, val_neg, device)\n    fa = len(count_fires(scores, 0.5, 1, eval_cfg.refractory_seconds))\n    hours = len(scores) / FRAMES_PER_HOUR\n    model.train()\n    return recall, fa / hours if hours else float("inf")\n\n\ndef train(\n    positives: np.ndarray,\n    adversarial: np.ndarray,\n    negative_stream: np.ndarray,\n    val_positives: np.ndarray,\n    val_negative_stream: np.ndarray,\n    cfg: TrainConfig,\n    eval_cfg: EvalConfig,\n    log=print,\n) -> tuple[WakeWordDNN, list[dict]]:\n    """positives/adversarial: [N, 16, 96]; negative streams: [T, 96]. All float16 or float32."""\n    torch.manual_seed(cfg.seed)\n    device = _device()\n    gen = torch.Generator(device=device).manual_seed(cfg.seed)\n\n    pos = torch.as_tensor(positives, dtype=torch.float16, device=device)\n    adv = torch.as_tensor(adversarial, dtype=torch.float16, device=device)\n    neg = torch.as_tensor(negative_stream, dtype=torch.float16, device=device)\n    val_neg = torch.as_tensor(val_negative_stream, dtype=torch.float16, device=device)\n    offsets = torch.arange(CLASSIFIER_FRAMES, device=device)\n\n    model = WakeWordDNN(cfg.layer_size, cfg.n_blocks).to(device)\n    opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rates[0])\n\n    phase_steps = [max(1, int(cfg.steps * f)) for f in cfg.phase_fractions]\n    total = sum(phase_steps)\n    eval_at = set(np.linspace(total // 2, total, cfg.eval_points, dtype=int).tolist())\n    y = torch.cat([\n        torch.ones(cfg.batch_positive, device=device),\n        torch.zeros(cfg.batch_adversarial + cfg.batch_negative, device=device),\n    ])\n    is_neg = y == 0\n\n    history: list[dict] = []\n    step, started = 0, time.time()\n    for lr, n_steps in zip(cfg.learning_rates, phase_steps):\n        for g in opt.param_groups:\n            g["lr"] = lr\n        for s in range(n_steps):\n            step += 1\n            neg_weight = 1.0 + (cfg.max_negative_weight - 1.0) * s / max(1, n_steps - 1)\n            p_idx = torch.randint(0, pos.shape[0], (cfg.batch_positive,), device=device, generator=gen)\n            a_idx = torch.randint(0, adv.shape[0], (cfg.batch_adversarial,), device=device, generator=gen)\n            n_idx = torch.randint(0, neg.shape[0] - CLASSIFIER_FRAMES + 1, (cfg.batch_negative,), device=device, generator=gen)\n            x = torch.cat([pos[p_idx], adv[a_idx], neg[n_idx[:, None] + offsets]]).float()\n\n            pred = model(x).squeeze(1).clamp(1e-7, 1 - 1e-7)\n            # Focus on examples still wrong enough to matter (as openWakeWord does).\n            mask = torch.where(is_neg, pred >= 0.001, pred < 0.999).float()\n            weights = torch.where(is_neg, torch.full_like(y, neg_weight), torch.ones_like(y))\n            loss = (F.binary_cross_entropy(pred, y, reduction="none") * weights * mask).sum() / mask.sum().clamp(min=1)\n\n            opt.zero_grad(set_to_none=True)\n            loss.backward()\n            opt.step()\n\n            if step in eval_at:\n                recall, fa_hr = _quick_metrics(model, val_positives, val_neg, device, eval_cfg)\n                history.append({"step": step, "loss": float(loss.detach()), "recall": recall, "fa_per_hour": fa_hr,\n                                "state": copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})})\n                log(f"step {step}/{total}  loss {float(loss.detach()):.4f}  recall@0.5 {recall:.3f}  FA/h@0.5 {fa_hr:.2f}  ({time.time() - started:.0f}s)")\n\n    if not history:\n        raise RuntimeError("no checkpoints were evaluated; increase steps")\n    pool = _candidates(history, cfg.average_top_k)\n    final = WakeWordDNN(cfg.layer_size, cfg.n_blocks).to(device)\n    choice = "single best checkpoint"\n    final.load_state_dict(pool[0]["state"])\n    best_single = (pool[0]["recall"], pool[0]["fa_per_hour"])\n    if len(pool) > 1:\n        averaged = WakeWordDNN(cfg.layer_size, cfg.n_blocks).to(device)\n        averaged.load_state_dict(average_state_dicts([h["state"] for h in pool]))\n        avg_metrics = _quick_metrics(averaged, val_positives, val_neg, device, eval_cfg)\n        if _objective(*avg_metrics) >= _objective(*best_single):\n            final, choice = averaged, f"average of {len(pool)} checkpoints"\n    recall, fa_hr = _quick_metrics(final, val_positives, val_neg, device, eval_cfg)\n    log(f"selected {choice}: recall@0.5 {recall:.3f}  FA/h@0.5 {fa_hr:.2f}")\n    return final.cpu().eval(), [{k: v for k, v in h.items() if k != "state"} for h in history]\n\n\ndef _objective(recall: float, fa_hr: float) -> float:\n    return recall - 0.02 * fa_hr\n\n\ndef _candidates(history: list[dict], top_k: int) -> list[dict]:\n    """Best checkpoints by recall, among those with at most the median false-accept rate; best first."""\n    median_fa = float(np.median([h["fa_per_hour"] for h in history]))\n    pool = [h for h in history if h["fa_per_hour"] <= median_fa] or history\n    return sorted(pool, key=lambda h: _objective(h["recall"], h["fa_per_hour"]), reverse=True)[:top_k]\n',
'wakeword_train/tts.py': '"""Synthetic speech with Piper (training-time only; nothing from Piper ships in the apps)."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom . import SAMPLE_RATE\nfrom .config import TtsConfig\nfrom .sources import to_16k_mono_int16\n\nMIN_PEAK = 1000  # int16; Piper normalises to full scale, so anything quieter is broken output\n\n\n@dataclass\nclass ClipSet:\n    """Variable-length int16 clips stored as one array plus offsets."""\n\n    audio: np.ndarray  # int16, all clips concatenated\n    offsets: np.ndarray  # int64, len = n + 1\n    speakers: np.ndarray  # int32 speaker id per clip\n    texts: np.ndarray  # int32 index into text_list per clip\n    text_list: list[str]\n\n    def __len__(self) -> int:\n        return len(self.offsets) - 1\n\n    def clip(self, i: int) -> np.ndarray:\n        return self.audio[self.offsets[i]:self.offsets[i + 1]]\n\n    def subset(self, mask: np.ndarray) -> "ClipSet":\n        idx = np.flatnonzero(mask)\n        clips = [self.clip(i) for i in idx]\n        return ClipSet.from_clips(clips, self.speakers[idx], self.texts[idx], self.text_list)\n\n    @staticmethod\n    def from_clips(clips: list[np.ndarray], speakers, texts, text_list: list[str]) -> "ClipSet":\n        lengths = np.array([len(c) for c in clips], dtype=np.int64)\n        offsets = np.concatenate([[0], np.cumsum(lengths)])\n        audio = np.concatenate(clips) if clips else np.empty(0, np.int16)\n        return ClipSet(audio.astype(np.int16), offsets, np.asarray(speakers, np.int32), np.asarray(texts, np.int32), list(text_list))\n\n    def save(self, path: Path) -> None:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        tmp = path.with_suffix(".tmp.npz")\n        np.savez(tmp, audio=self.audio, offsets=self.offsets, speakers=self.speakers, texts=self.texts,\n                 text_list=np.array(self.text_list, dtype=object))\n        tmp.replace(path)\n\n    @staticmethod\n    def load(path: Path) -> "ClipSet":\n        z = np.load(path, allow_pickle=True)\n        return ClipSet(z["audio"], z["offsets"], z["speakers"], z["texts"], list(z["text_list"]))\n\n\ndef trim_silence(audio: np.ndarray, threshold_db: float = -40.0, pad_seconds: float = 0.05) -> np.ndarray:\n    """Trims leading/trailing audio quieter than threshold_db relative to the clip\'s peak (20 ms blocks)."""\n    block = SAMPLE_RATE // 50\n    n = len(audio) // block\n    if n == 0:\n        return audio\n    x = audio[: n * block].astype(np.float32).reshape(n, block)\n    rms = np.sqrt((x**2).mean(axis=1) + 1e-9)\n    loud = np.flatnonzero(20 * np.log10(rms / (rms.max() + 1e-9)) > threshold_db)\n    if loud.size == 0:\n        return audio\n    pad = int(pad_seconds * SAMPLE_RATE)\n    start = max(0, loud[0] * block - pad)\n    end = min(len(audio), (loud[-1] + 1) * block + pad)\n    return audio[start:end]\n\n\nclass PiperSynth:\n    def __init__(self, model_path: Path, config_path: Path, use_cuda: bool = False):\n        from piper import PiperVoice  # imported lazily: only the TTS steps need it\n\n        self.voice = PiperVoice.load(model_path, config_path=config_path, use_cuda=use_cuda)\n        self.sample_rate = self.voice.config.sample_rate\n        self.num_speakers = max(1, self.voice.config.num_speakers)\n\n    def synth(self, text: str, speaker_id: int, length_scale: float, noise_scale: float, noise_w_scale: float) -> np.ndarray:\n        from piper import SynthesisConfig\n\n        cfg = SynthesisConfig(\n            speaker_id=speaker_id if self.num_speakers > 1 else None,\n            length_scale=length_scale,\n            noise_scale=noise_scale,\n            noise_w_scale=noise_w_scale,\n            normalize_audio=True,\n        )\n        chunks = [c.audio_float_array for c in self.voice.synthesize(text, syn_config=cfg)]\n        audio = np.concatenate(chunks) if chunks else np.zeros(0, np.float32)\n        return trim_silence(to_16k_mono_int16(audio, self.sample_rate))\n\n\ndef generate(\n    synths: list[PiperSynth],\n    texts: list[str],\n    n: int,\n    cfg: TtsConfig,\n    rng: np.random.Generator,\n    progress=None,\n    speaker_offset: int = 0,\n) -> ClipSet:\n    """n clips cycling through texts, spread across voices and speakers, including held-out speakers.\n\n    Speaker ids are offset per voice (voice k uses ids k*100000 + id) so holdout is tracked across voices.\n    """\n    clips, speakers, text_idx = [], [], []\n    for i in range(n):\n        v = i % len(synths)\n        synth = synths[v]\n        speaker = int(rng.integers(0, synth.num_speakers))\n        t = i % len(texts)\n        audio = synth.synth(\n            texts[t],\n            speaker,\n            float(rng.uniform(*cfg.length_scale)),\n            float(rng.uniform(*cfg.noise_scale)),\n            float(rng.uniform(*cfg.noise_w_scale)),\n        )\n        if len(audio) < SAMPLE_RATE // 10 or np.abs(audio.astype(np.int32)).max() < MIN_PEAK:\n            continue  # empty or near-silent output is never a useful example\n        clips.append(audio)\n        speakers.append(speaker_offset + v * 100000 + speaker)\n        text_idx.append(t)\n        if progress:\n            progress(i + 1, n)\n    return ClipSet.from_clips(clips, speakers, text_idx, texts)\n\n\ndef is_held_out(speakers: np.ndarray, holdout_every_nth: int) -> np.ndarray:\n    return (speakers % 100000) % holdout_every_nth == 0\n\n\nGROUP_SPEAKERS = 1_000_000  # speaker ids of voice group g start at g * GROUP_SPEAKERS\n\n\ndef held_out_mask(speakers: np.ndarray, cfg: TtsConfig) -> np.ndarray:\n    """Validation speakers: every nth speaker, except in voice groups held out entirely or not at all."""\n    mask = is_held_out(speakers, cfg.holdout_every_nth_speaker)\n    for g, grp in enumerate(cfg.groups, start=1):\n        sel = speakers // GROUP_SPEAKERS == g\n        if grp.holdout == "all":\n            mask[sel] = True\n        elif grp.holdout == "none":\n            mask[sel] = False\n    return mask\n\n\n# Parler-TTS Mini v1\'s named speakers keep the same voice across generations.\nPARLER_NAMES = ["Laura", "Gary", "Jon", "Lea", "Karen", "Rick", "Brenda", "David", "Eileen", "Jordan", "Mike", "Yann",\n                "Joy", "James", "Eric", "Lauren", "Rose", "Will", "Jason", "Aaron", "Naomie", "Alisa", "Patrick",\n                "Jerry", "Tina", "Jenna", "Bill", "Tom", "Carol", "Barbara", "Rebecca", "Anna", "Bruce", "Emily"]\n\n\ndef parler_descriptions(n: int, seed: int = 0) -> list[str]:\n    """n speaker descriptions: the named speakers first, then voices described by gender, pitch, pace and room."""\n    rng = np.random.default_rng(seed)\n    pitch = ["a low-pitched", "a slightly low-pitched", "a moderate-pitched", "a slightly high-pitched", "a high-pitched"]\n    pace = ["slowly", "at a moderate pace", "quite fast", "fast"]\n    style = ["in a monotone way", "with a slightly expressive tone", "in an animated, expressive way", "casually"]\n    room = ["in a very close-sounding environment", "in a confined room", "in a room with some echo",\n            "from a little distance, in a large room"]\n    quality = ["The recording is very clear.", "The recording is clear.", "There is a little background noise.",\n               "The recording is slightly noisy."]\n    out = []\n    for i in range(n):\n        p, a, s, r, q = (x[int(rng.integers(len(x)))] for x in (pitch, pace, style, room, quality))\n        if i < len(PARLER_NAMES):\n            out.append(f"{PARLER_NAMES[i]} speaks {s} {a} {r}. {q}")\n        else:\n            who = "female" if rng.random() < 0.5 else "male"\n            out.append(f"A {who} speaker with {p} voice speaks {s} {a} {r}. {q}")\n    return out\n\n\nclass ParlerSynth:\n    """Parler-TTS (Apache-2.0; trained from scratch on LibriTTS-R and English MLS, both CC BY 4.0).\n\n    Speakers are text descriptions, so the number of voices is limited only by how many descriptions are drawn.\n    Generation is capped at max_seconds: a batch runs until every clip in it ends, and Parler sometimes rambles on\n    after short text up to its 30 s limit, which made a T4 take half an hour per 2000 clips.\n    """\n\n    max_seconds = 5.0  # wake phrases and near-misses take under 3 s\n\n    def __init__(self, model_id: str, descriptions: list[str], use_cuda: bool = False):\n        import torch\n        from parler_tts import ParlerTTSForConditionalGeneration  # only the Parler voice group needs it\n        from transformers import AutoTokenizer\n\n        self.torch = torch\n        self.device = "cuda" if use_cuda and torch.cuda.is_available() else "cpu"\n        self.model = ParlerTTSForConditionalGeneration.from_pretrained(model_id).to(self.device).eval()\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id)\n        self.sample_rate = self.model.config.sampling_rate\n        # Decoder steps: one per audio frame, plus one per codebook for the delay pattern.\n        frame_rate = self.model.audio_encoder.config.frame_rate\n        self.max_new_tokens = int(self.max_seconds * frame_rate) + self.model.decoder.config.num_codebooks\n        self.descriptions = descriptions\n        self.num_speakers = len(descriptions)\n\n    def synth_batch(self, items: list[tuple[str, int]], seed: int) -> list[np.ndarray]:\n        """(text, speaker) pairs -> int16 16 kHz clips, generated in one batch."""\n        torch = self.torch\n        desc = self.tokenizer([self.descriptions[s] for _, s in items], return_tensors="pt", padding=True).to(self.device)\n        prompt = self.tokenizer([t for t, _ in items], return_tensors="pt", padding=True).to(self.device)\n        torch.manual_seed(seed)\n        with torch.inference_mode():\n            gen = self.model.generate(\n                input_ids=desc.input_ids, attention_mask=desc.attention_mask,\n                prompt_input_ids=prompt.input_ids, prompt_attention_mask=prompt.attention_mask,\n                do_sample=True, return_dict_in_generate=True, max_new_tokens=self.max_new_tokens,\n            )\n        lengths = getattr(gen, "audios_length", None)\n        out: list[np.ndarray] = []\n        for i in range(len(items)):\n            audio = gen.sequences[i]\n            if lengths is not None:\n                audio = audio[: int(lengths[i])]\n            out.append(trim_silence(to_16k_mono_int16(audio.float().cpu().numpy(), self.sample_rate)))\n        return out\n\n\ndef generate_parler(synth: ParlerSynth, texts: list[str], n: int, rng: np.random.Generator,\n                    speaker_offset: int = 0, batch: int = 24, progress=None) -> ClipSet:\n    """Like generate(), for Parler-TTS: n clips cycling through texts, random described speakers, in batches."""\n    clips, speakers, text_idx = [], [], []\n    for start in range(0, n, batch):\n        items = [(i % len(texts), int(rng.integers(synth.num_speakers))) for i in range(start, min(start + batch, n))]\n        audios = synth.synth_batch([(texts[t], s) for t, s in items], int(rng.integers(2**31)))\n        for (t, s), audio in zip(items, audios):\n            if len(audio) < SAMPLE_RATE // 10 or np.abs(audio.astype(np.int32)).max() < MIN_PEAK:\n                continue\n            if len(audio) > (getattr(synth, "max_seconds", 5.0) - 0.5) * SAMPLE_RATE:\n                continue  # ran into the length cap: it rambled on past the text\n            clips.append(audio)\n            speakers.append(speaker_offset + s)\n            text_idx.append(t)\n        if progress:\n            progress(min(start + batch, n), n)\n    return ClipSet.from_clips(clips, speakers, text_idx, texts)\n'
}
for rel, text in FILES.items():
    path = os.path.join('/content/training', rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, 'w').write(text)
os.makedirs('/content/training/configs', exist_ok=True)  # the %%writefile cell below cannot create folders
print(f'wrote {len(FILES)} files')

## Configuration
Edit here if needed, for example the near-miss phrases or `positives_per_phrase`. The defaults are the v0 run from the plan.

In [ ]:
%%writefile /content/training/configs/uno.yaml
# v0 models for "Hey UNO" and "Hello UNO". Every source here allows commercial use.
# Positives are synthetic only; retrain with real recordings (plan 5.2) before release.
run_name: uno-v0

phrases:
  - id: hey_uno
    display: Hey UNO
    # Spellings that make Piper say "oo-no". Check the preview step and edit if it says "you-no" or spells letters.
    tts_texts: ["hey uno", "hey, uno", "hey oono", "hey uno!", "hey ooh-no"]
    adversarial_texts:
      - hey you know
      - you know
      - hey, you know what
      - you know what I mean
      - hey you
      - hey you now
      - hey you go
      - hey juno
      - hey bruno
      - hey no
      - hey new
      - hey yo
      - hey there
      - okay you know
      - uno
      - let's play uno
      - hey, one
      - hey, number one
    transcript_variants: [hey you know, hey you no, hey juno, hey oono, hey u no, hey uno]
  - id: hello_uno
    display: Hello UNO
    tts_texts: ["hello uno", "hello, uno", "hello oono", "hello uno!", "hello ooh-no"]
    adversarial_texts:
      - hello
      - hello you know
      - hello, you know what
      - hello you
      - hello juno
      - hello bruno
      - hello no
      - hello new
      - hello there
      - yellow uno
      - below you know
      - hollow uno
      - uno
      - let's play uno
      - hello, one
      - oh hello you
    transcript_variants: [hello you know, hello you no, hello juno, hello oono, hello u no, hello uno]

tts:
  voices:
    - name: en_US-libritts_r-medium  # 904 speakers
      model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx
      config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json
      # MODEL_CARD: trained on LibriTTS-R (openslr.org/141, CC BY 4.0), but fine-tuned from Piper's lessac voice,
      # whose Blizzard 2013 recordings are licensed for research. Whether that carries over needs legal review.
      license: CC-BY-4.0
      license_url: https://creativecommons.org/licenses/by/4.0/
  # More voices, each group with clips of its own; a finished run generates only the groups it lacks.
  groups:
    # Piper voices trained from scratch on public-domain recordings: four to train on ...
    - name: piper_public_domain
      engine: piper
      clips_per_phrase: 4000
      holdout: none
      voices:
        - name: en_GB-cori-medium  # UK female, LibriVox
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/cori/medium/en_GB-cori-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/cori/medium/en_GB-cori-medium.onnx.json
          license: Public-Domain
          license_url: https://librivox.org/pages/public-domain/
        - name: en_US-kristin-medium  # US female, LibriVox
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/kristin/medium/en_US-kristin-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/kristin/medium/en_US-kristin-medium.onnx.json
          license: Public-Domain
          license_url: https://librivox.org/pages/public-domain/
        - name: en_US-john-medium  # US male, LibriVox; fine-tuned from kristin
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/john/medium/en_US-john-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/john/medium/en_US-john-medium.onnx.json
          license: Public-Domain
          license_url: https://librivox.org/pages/public-domain/
        - name: en_US-norman-medium  # US male, LibriVox
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/norman/medium/en_US-norman-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/norman/medium/en_US-norman-medium.onnx.json
          license: Public-Domain
          license_url: https://librivox.org/pages/public-domain/
    # ... and one never trained on, to measure how the models do on a voice they have not heard.
    - name: piper_unseen
      engine: piper
      clips_per_phrase: 1000
      holdout: all
      voices:
        - name: en_US-ljspeech-medium  # US female, LJ Speech
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ljspeech/medium/en_US-ljspeech-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ljspeech/medium/en_US-ljspeech-medium.onnx.json
          license: Public-Domain
          license_url: https://keithito.com/LJ-Speech-Dataset/
    # Parler-TTS Mini v1: Apache-2.0 weights trained from scratch on LibriTTS-R and English MLS (both CC BY 4.0).
    # Speakers are text descriptions (its 34 named voices, then gender, pitch, pace and room), so hundreds of voices.
    - name: parler
      engine: parler
      clips_per_phrase: 4000
      speakers: 400
      holdout: speakers
      voices:
        - name: parler-tts-mini-v1
          model_url: parler-tts/parler-tts-mini-v1
          config_url: ""
          license: Apache-2.0
          license_url: https://huggingface.co/parler-tts/parler-tts-mini-v1
    # Accents (Scottish, Irish, Indian, Canadian and more: about 150 speakers), but every one is fine-tuned from
    # Piper's lessac voice, like the libritts_r voice above. Off until legal clears the lessac question.
    - name: piper_lessac_based
      enabled: false
      engine: piper
      clips_per_phrase: 6000
      holdout: speakers
      voices:
        - name: en_GB-vctk-medium  # 109 speakers, UK and other accents
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/vctk/medium/en_GB-vctk-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/vctk/medium/en_GB-vctk-medium.onnx.json
          license: CC-BY-4.0
          license_url: https://datashare.ed.ac.uk/handle/10283/3443
        - name: en_GB-aru-medium  # 12 UK speakers
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/aru/medium/en_GB-aru-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/aru/medium/en_GB-aru-medium.onnx.json
          license: CC-BY-4.0
          license_url: https://www.liverpool.ac.uk/architecture/research/acoustics-research-unit/speech-corpus/
        - name: en_GB-alba-medium  # Scottish female
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/alba/medium/en_GB-alba-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/alba/medium/en_GB-alba-medium.onnx.json
          license: CC-BY-4.0
          license_url: https://datashare.ed.ac.uk/handle/10283/3270
        - name: en_US-joe-medium  # US male
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/joe/medium/en_US-joe-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/joe/medium/en_US-joe-medium.onnx.json
          license: CC0-1.0
          license_url: https://github.com/OHF-Voice/voice-datasets
        - name: en_US-mike-medium  # US male
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/mike/medium/en_US-mike-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/mike/medium/en_US-mike-medium.onnx.json
          license: CC0-1.0
          license_url: https://github.com/OHF-Voice/voice-datasets
        - name: en_US-sam-medium  # non-binary voice
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/sam/medium/en_US-sam-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/sam/medium/en_US-sam-medium.onnx.json
          license: Apache-2.0
          license_url: https://github.com/Sam-Accenture-Non-Binary-Voice/non-binary-voice-files
        - name: en_US-reza_ibrahim-medium  # male, Middle Eastern accent
          model_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/reza_ibrahim/medium/en_US-reza_ibrahim-medium.onnx
          config_url: https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/reza_ibrahim/medium/en_US-reza_ibrahim-medium.onnx.json
          license: CC0-1.0
          license_url: https://huggingface.co/datasets/mah92/Ibrahim-Walk-English-Quran-Audio-Dataset
  positives_per_phrase: 20000
  adversarial_per_phrase: 20000
  use_cuda: true

negatives:
  - name: librispeech_train
    kind: tar
    split: train
    max_hours: 300
    urls:
      - https://www.openslr.org/resources/12/train-clean-100.tar.gz
      - https://www.openslr.org/resources/12/train-other-500.tar.gz
      - https://www.openslr.org/resources/12/train-clean-360.tar.gz
    license: CC-BY-4.0
    license_url: https://creativecommons.org/licenses/by/4.0/
    source_url: https://www.openslr.org/12/
  - name: musan
    kind: tar
    split: train
    max_hours: 108.8  # all of it: the three folders hold 108.9 h, so a higher cap would re-stream 11 GB for nothing
    urls: [https://www.openslr.org/resources/17/musan.tar.gz]
    include: [musan/music/, musan/noise/, musan/speech/]
    noise_bank: [musan/noise/, musan/music/]
    # OpenSLR lists CC BY 4.0; the corpus excludes NonCommercial files. Per-file attribution lives in MUSAN's LICENSE files.
    license: CC-BY-4.0
    license_url: https://creativecommons.org/licenses/by/4.0/
    source_url: https://www.openslr.org/17/
  # Conversation, so the model learns not to fire on everyday talk (validation alone only measured it).
  # None of these recordings is in validation: other VoxPopuli year, other AMI meetings.
  - name: voxpopuli_train
    kind: tar
    split: train
    max_hours: 60
    urls: [https://dl.fbaipublicfiles.com/voxpopuli/audios/en_2019.tar]
    download_first: true  # 6.9 GB; see voxpopuli_val
    license: CC0-1.0
    license_url: https://creativecommons.org/publicdomain/zero/1.0/
    source_url: https://github.com/facebookresearch/voxpopuli
  - name: ami_train
    # The 33 non-scenario meetings (EN, IB, IN; about 27 h), each twice: the far-field table microphone and
    # the close-talk headset mix, like a device across the table and a phone held close.
    kind: files
    split: train
    max_hours: 55
    urls:
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001a/audio/EN2001a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001b/audio/EN2001b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001d/audio/EN2001d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001e/audio/EN2001e.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002a/audio/EN2002a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002b/audio/EN2002b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002c/audio/EN2002c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002d/audio/EN2002d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2003a/audio/EN2003a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2004a/audio/EN2004a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2005a/audio/EN2005a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2006a/audio/EN2006a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2006b/audio/EN2006b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2009b/audio/EN2009b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2009c/audio/EN2009c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2009d/audio/EN2009d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4001/audio/IB4001.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4002/audio/IB4002.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4003/audio/IB4003.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4004/audio/IB4004.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4005/audio/IB4005.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4010/audio/IB4010.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4011/audio/IB4011.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1001/audio/IN1001.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1002/audio/IN1002.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1005/audio/IN1005.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1007/audio/IN1007.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1008/audio/IN1008.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1009/audio/IN1009.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1012/audio/IN1012.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1013/audio/IN1013.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1014/audio/IN1014.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1016/audio/IN1016.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001a/audio/EN2001a.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001b/audio/EN2001b.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001d/audio/EN2001d.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2001e/audio/EN2001e.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002a/audio/EN2002a.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002b/audio/EN2002b.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002c/audio/EN2002c.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2002d/audio/EN2002d.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2003a/audio/EN2003a.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2004a/audio/EN2004a.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2005a/audio/EN2005a.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2006a/audio/EN2006a.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2006b/audio/EN2006b.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2009b/audio/EN2009b.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2009c/audio/EN2009c.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/EN2009d/audio/EN2009d.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4001/audio/IB4001.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4002/audio/IB4002.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4003/audio/IB4003.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4004/audio/IB4004.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4005/audio/IB4005.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4010/audio/IB4010.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IB4011/audio/IB4011.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1001/audio/IN1001.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1002/audio/IN1002.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1005/audio/IN1005.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1007/audio/IN1007.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1008/audio/IN1008.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1009/audio/IN1009.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1012/audio/IN1012.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1013/audio/IN1013.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1014/audio/IN1014.Mix-Headset.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IN1016/audio/IN1016.Mix-Headset.wav
    license: CC-BY-4.0
    license_url: https://creativecommons.org/licenses/by/4.0/
    source_url: https://groups.inf.ed.ac.uk/ami/corpus/
  # Your own internal meetings, whose participants agreed to training use: features (not audio) exported on your
  # machine by `python -m wakeword_train.test_package ... --train-share 0.67 --export-train <folder>`, with that folder
  # uploaded here. The rest of the meetings stay a frozen local test set. Skipped when the folder is missing.
  - name: meetings_train
    kind: embeddings
    split: train
    max_hours: 100
    urls: [/content/drive/MyDrive/wakeword/meetings_train]
    license: Internal-Consent
    # The consent email sent to meeting participants (subject starts with the release_id), and their replies.
    license_url: "internal: email 'WW-MEET-2026-10-07: wake word model training on recorded internal meetings'"
    release_id: "WW-MEET-2026-10-07"
  - name: librispeech_val
    kind: tar
    split: val
    max_hours: 30
    urls:
      - https://www.openslr.org/resources/12/dev-clean.tar.gz
      - https://www.openslr.org/resources/12/dev-other.tar.gz
      - https://www.openslr.org/resources/12/test-clean.tar.gz
      - https://www.openslr.org/resources/12/test-other.tar.gz
    license: CC-BY-4.0
    license_url: https://creativecommons.org/licenses/by/4.0/
    source_url: https://www.openslr.org/12/
  # Harder validation audio (plan Section 3: TV, podcasts, conversation, ambient), so calibration is not
  # done on audiobooks alone. Validation only: none of it is trained on.
  - name: voxpopuli_val
    kind: tar
    split: val
    max_hours: 40
    urls: [https://dl.fbaipublicfiles.com/voxpopuli/audios/en_2020.tar]  # whole European Parliament sessions
    download_first: true  # 5.2 GB; the server resets streams left idle during hours-long sessions
    license: CC0-1.0
    license_url: https://creativecommons.org/publicdomain/zero/1.0/
    source_url: https://github.com/facebookresearch/voxpopuli
  - name: ami_val
    # Real meetings heard by one far-field table microphone, like a laptop or phone on the desk: 136 scenario
    # meetings (ES, IS, TS), about 72 h. The EN, IB and IN meetings are left for training negatives.
    kind: files
    split: val
    max_hours: 72
    urls:
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2002a/audio/ES2002a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2002b/audio/ES2002b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2002c/audio/ES2002c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2002d/audio/ES2002d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2003a/audio/ES2003a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2003b/audio/ES2003b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2003c/audio/ES2003c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2003d/audio/ES2003d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004a/audio/ES2004a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004b/audio/ES2004b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004c/audio/ES2004c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004d/audio/ES2004d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2005a/audio/ES2005a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2005b/audio/ES2005b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2005c/audio/ES2005c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2005d/audio/ES2005d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2006a/audio/ES2006a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2006b/audio/ES2006b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2006c/audio/ES2006c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2006d/audio/ES2006d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2007a/audio/ES2007a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2007b/audio/ES2007b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2007c/audio/ES2007c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2007d/audio/ES2007d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2008a/audio/ES2008a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2008b/audio/ES2008b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2008c/audio/ES2008c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2008d/audio/ES2008d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2009a/audio/ES2009a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2009b/audio/ES2009b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2009c/audio/ES2009c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2009d/audio/ES2009d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2010a/audio/ES2010a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2010b/audio/ES2010b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2010c/audio/ES2010c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2010d/audio/ES2010d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2011a/audio/ES2011a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2011b/audio/ES2011b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2011c/audio/ES2011c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2011d/audio/ES2011d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2012a/audio/ES2012a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2012b/audio/ES2012b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2012c/audio/ES2012c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2012d/audio/ES2012d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2013a/audio/ES2013a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2013b/audio/ES2013b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2013c/audio/ES2013c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2013d/audio/ES2013d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2014a/audio/ES2014a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2014b/audio/ES2014b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2014c/audio/ES2014c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2014d/audio/ES2014d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2015a/audio/ES2015a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2015b/audio/ES2015b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2015c/audio/ES2015c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2015d/audio/ES2015d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2016a/audio/ES2016a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2016b/audio/ES2016b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2016c/audio/ES2016c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2016d/audio/ES2016d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1000a/audio/IS1000a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1000b/audio/IS1000b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1000c/audio/IS1000c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1000d/audio/IS1000d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1001a/audio/IS1001a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1001b/audio/IS1001b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1001c/audio/IS1001c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1001d/audio/IS1001d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1002b/audio/IS1002b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1002c/audio/IS1002c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1002d/audio/IS1002d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1003a/audio/IS1003a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1003c/audio/IS1003c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1003d/audio/IS1003d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1004a/audio/IS1004a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1004b/audio/IS1004b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1004c/audio/IS1004c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1004d/audio/IS1004d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1005a/audio/IS1005a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1005b/audio/IS1005b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1005c/audio/IS1005c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1006a/audio/IS1006a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1006b/audio/IS1006b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1006c/audio/IS1006c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1006d/audio/IS1006d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1007a/audio/IS1007a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1007b/audio/IS1007b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1007c/audio/IS1007c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1008a/audio/IS1008a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1008b/audio/IS1008b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1008c/audio/IS1008c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1008d/audio/IS1008d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1009a/audio/IS1009a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1009b/audio/IS1009b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1009c/audio/IS1009c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/IS1009d/audio/IS1009d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3003a/audio/TS3003a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3003b/audio/TS3003b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3003c/audio/TS3003c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3003d/audio/TS3003d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3004a/audio/TS3004a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3004b/audio/TS3004b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3004c/audio/TS3004c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3004d/audio/TS3004d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3005a/audio/TS3005a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3005b/audio/TS3005b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3005c/audio/TS3005c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3005d/audio/TS3005d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3006a/audio/TS3006a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3006b/audio/TS3006b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3006c/audio/TS3006c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3006d/audio/TS3006d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3007a/audio/TS3007a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3007b/audio/TS3007b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3007c/audio/TS3007c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3007d/audio/TS3007d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3008a/audio/TS3008a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3008b/audio/TS3008b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3008c/audio/TS3008c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3008d/audio/TS3008d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3009a/audio/TS3009a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3009b/audio/TS3009b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3009c/audio/TS3009c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3009d/audio/TS3009d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3010a/audio/TS3010a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3010b/audio/TS3010b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3010c/audio/TS3010c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3010d/audio/TS3010d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3011a/audio/TS3011a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3011b/audio/TS3011b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3011c/audio/TS3011c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3011d/audio/TS3011d.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3012a/audio/TS3012a.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3012b/audio/TS3012b.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3012c/audio/TS3012c.Array1-01.wav
      - https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/TS3012d/audio/TS3012d.Array1-01.wav
    license: CC-BY-4.0
    license_url: https://creativecommons.org/licenses/by/4.0/
    source_url: https://groups.inf.ed.ac.uk/ami/corpus/

rir_url: https://mcdermottlab.mit.edu/Reverb/IRMAudio/Audio.zip
rir:
  license: CC-BY-4.0
  license_url: https://creativecommons.org/licenses/by/4.0/

melspectrogram_url: https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx
embedding_url: https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx
features_use_cuda: true
feature_models:
  license: Apache-2.0  # openWakeWord; embedding derived from Google's Apache-2.0 speech_embedding
  license_url: https://github.com/dscripka/openWakeWord/blob/main/LICENSE

augment:
  copies_per_clip: 2
  noise_bank_hours: 6

train:
  steps: 40000

eval:
  consecutive_frames: 3
  refractory_seconds: 1.5

In [ ]:
#@title 1. Setup
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps setup

In [ ]:
#@title 2. Preview pronunciation
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps preview
import glob
from IPython.display import Audio, display
for f in sorted(glob.glob(f'{WORK}/preview/*.wav')):
    print(os.path.basename(f)); display(Audio(f))

**Listen to the previews.** Each should sound like "Hey OO-noh" or "Hello OO-noh". If a spelling sounds wrong, remove it from `tts_texts` in the configuration cell, run that cell again, then delete `steps/preview.done` in the Drive folder and re-run the preview.

In [ ]:
#@title 3. Generate voices
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps tts

In [ ]:
#@title 4. Background audio (longest step; resumes if interrupted)
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps negatives

In [ ]:
#@title 5. Features
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps features

In [ ]:
#@title 6. Train
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps train

In [ ]:
#@title 7. Evaluate
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps evaluate
print(open(f'{WORK}/steps/evaluate.done').read())

In [ ]:
#@title 8. Export and download
!cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" --steps export
from google.colab import files
zip_path = f'{WORK}/export/wakeword_models.zip'
print(open(f'{WORK}/export/wakeword_models/METRICS.md').read())
files.download(zip_path)

In [ ]:
#@title 9. (Optional) Pack the base for Wake Word Studio
# Wake Word Studio trains new phrases on your own machine, reusing this run's background audio,
# noise and models. This writes one file to download: about 4-5 GB, so it needs that much free Drive space.
PACK_STUDIO_BASE = False  #@param {type:"boolean"}
import os, shutil, tarfile
out = f'{WORK}/studio_base.tar'
if not PACK_STUDIO_BASE:
    print('Skipped. Tick PACK_STUDIO_BASE and run this cell to pack the Studio base.')
elif not os.path.exists(out):
    # Build on the Colab disk, then copy once: writing GBs into the Drive mount piecemeal can drop it.
    local = '/content/studio_base.tar'
    with tarfile.open(local, 'w') as t:
        for rel in ['assets', 'noise/bank.npy', 'negatives']:
            t.add(os.path.join(WORK, rel), arcname=f'studio_base/{rel}')
    shutil.copy(local, out + '.tmp')
    os.replace(out + '.tmp', out)
    os.remove(local)
if os.path.exists(out):
    print(f'{out}: {os.path.getsize(out) / 2**30:.1f} GB. Download it from Google Drive (wakeword/{RUN}/studio_base.tar).')

## (Optional) Recalibrate a finished run
Only for a run trained before the validation audio was extended (meetings and parliamentary sessions, not just audiobooks), such as `uno-v1`. New runs already include it, so skip this.

Set `RUN` in the first cell to that run, run the cells down to *Configuration*, then tick **RECALIBRATE** below and run it. It downloads only the missing validation audio (about 110 hours), re-measures both models, recalibrates sensitivity and downloads a new zip. No retraining: about 30–60 minutes.

In [ ]:
#@title 10. (Optional) Recalibrate on the extended validation audio
RECALIBRATE = False  #@param {type:"boolean"}
if RECALIBRATE:
    import time
    started = time.time()
    get_ipython().system('cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" '
                         '--steps negatives,evaluate,export --force --negatives-split val')
    zip_path = f'{WORK}/export/wakeword_models.zip'
    if not os.path.exists(zip_path) or os.path.getmtime(zip_path) < started:
        raise RuntimeError('Recalibration stopped before it finished (see the output above), so there is no new zip. '
                           'Run this cell again: it resumes where it stopped.')
    from google.colab import files
    print(open(f'{WORK}/export/wakeword_models/METRICS.md').read())
    files.download(zip_path)
else:
    print('Skipped. Tick RECALIBRATE and run this cell to recalibrate a finished run.')

## (Optional) Retrain a finished run with conversational background audio
For a finished run (such as `uno-v1`) whose training data has grown since: new background audio (the meeting and parliament recordings, or your own meetings' features in `MyDrive/wakeword/meetings_train`), or new voice groups under `tts: groups:` in the configuration. Only what the run lacks is generated; the voices it already has are kept. New runs already include whatever is configured.

**Needs a T4 GPU.** Set `RUN`, run the cells down to *Configuration*, then tick **RETRAIN** below and run it. It fetches only the training audio and voice clips the run does not have yet, rebuilds the features if the voices changed, retrains both models, re-measures them and downloads a new zip: 20 minutes to about 3 hours depending on how much is new (new voice groups take the longest). The models it replaces are kept in `models/previous/<date-time>/`.

In [ ]:
#@title 11. (Optional) Retrain with conversational background audio
RETRAIN = False  #@param {type:"boolean"}
if RETRAIN:
    import glob, shutil, time, torch
    if not torch.cuda.is_available():
        raise RuntimeError('Retraining needs a GPU: Runtime -> Change runtime type -> T4 GPU, then run the cells from the top.')
    started = time.time()
    marker = f'{WORK}/models/RETRAINING'
    if not os.path.exists(marker):  # a new retrain sets the current models aside; a re-run after a stop resumes instead
        previous = f'{WORK}/models/previous/{time.strftime("%Y%m%d-%H%M%S")}'
        os.makedirs(previous, exist_ok=True)
        for f in glob.glob(f'{WORK}/models/*.pt') + glob.glob(f'{WORK}/models/*_history.json'):
            shutil.move(f, previous)
        open(marker, 'w').close()
    get_ipython().system('cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" '
                         '--steps negatives,tts,features,train,evaluate,export --force --negatives-split train')
    zip_path = f'{WORK}/export/wakeword_models.zip'
    if not os.path.exists(zip_path) or os.path.getmtime(zip_path) < started:
        raise RuntimeError('Retraining stopped before it finished (see the output above), so there is no new zip. '
                           'Run this cell again: it resumes where it stopped.')
    os.remove(marker)
    from google.colab import files
    print(open(f'{WORK}/export/wakeword_models/METRICS.md').read())
    files.download(zip_path)
else:
    print('Skipped. Tick RETRAIN and run this cell to retrain a finished run.')

## (Optional) Test on your own recordings
Measures how often the models fire by mistake on your own audio, such as team meetings from Google Meet or Teams. They are only measured on, never trained or calibrated on, so the result stays an honest real-world estimate (plan 5.3).

1. **Only recordings whose participants agreed to this use.** Leave out meetings with outside people unless they agreed too, and any meeting where someone actually says the wake phrase.
2. Put the files in one Drive folder (subfolders are fine). Meet saves `.mp4` to *My Drive → Meet Recordings*; download Teams recordings from OneDrive or SharePoint. Copy the ones you can use rather than pointing at the whole Meet folder.
3. Fill in the form, tick **TEST** and run the cell. About 10 minutes per 30 hours on a T4, about an hour on CPU.

Only feature numbers are kept, never the audio. The zip gets totals only; `test_fires.csv` (file and time of each fire, to check by ear) is downloaded separately and is not part of the package.

In [ ]:
#@title 12. (Optional) Test on your own recordings
TEST_FOLDER = '/content/drive/MyDrive/wakeword/meetings'  #@param {type:"string"}
CONSENT_URL = ''  #@param {type:"string"}
CONSENT_ID = ''  #@param {type:"string"}
TEST = False  #@param {type:"boolean"}
if TEST:
    import shlex, time
    if not os.path.isdir(TEST_FOLDER):
        raise RuntimeError(f'No folder {TEST_FOLDER}: create it in Google Drive and put the recordings in it.')
    started = time.time()
    sets = {'test.folder': TEST_FOLDER, 'test.license_url': CONSENT_URL, 'test.release_id': CONSENT_ID}
    args = ' '.join(f'--set {shlex.quote(f"{k}={v}")}' for k, v in sets.items() if v)
    get_ipython().system('cd /content/training && python -m wakeword_train.pipeline --config configs/uno.yaml --work "$WORK" '
                         f'--steps test,export --force {args}')
    zip_path = f'{WORK}/export/wakeword_models.zip'
    if not os.path.exists(zip_path) or os.path.getmtime(zip_path) < started:
        raise RuntimeError('The test stopped before it finished (see the output above). Run this cell again: '
                           'recordings already processed are not processed again.')
    if not (CONSENT_URL and CONSENT_ID):
        print('Note: without CONSENT_URL and CONSENT_ID, DATASET_MANIFEST.csv will fail the license check '
              '(python -m data.manifest). Fill them in and run this cell again before sharing the package.')
    from google.colab import files
    print(open(f'{WORK}/export/wakeword_models/METRICS.md').read())
    files.download(zip_path)
    files.download(f'{WORK}/eval/test_fires.csv')
else:
    print('Skipped. Fill in the form, tick TEST and run this cell to test on your own recordings.')

## What you get
`wakeword_models.zip` holds `melspectrogram.onnx`, `embedding_model.onnx`, `hey_uno.onnx`, `hello_uno.onnx`, `models.json` (keywords, sensitivity tables and detector settings), `golden.json` (parity vectors for the app), `METRICS.md` and `DATASET_MANIFEST.csv`. Copy it into the repo as `models/` and share `METRICS.md` with the team.

These are **v0** models trained on synthetic voices. Retrain with the real recordings (plan 5.2) before release.